In [1]:
import os
import json
import base64
import numpy as np
import pandas as pd
import geopandas as gpd
from typing import Optional
from pyiceberg.expressions import And, GreaterThanOrEqual, LessThanOrEqual
from pyiceberg.catalog import load_catalog

In [64]:
us_cbsa = gpd.read_file("/Users/houpuli/Downloads/Core_based_statistical_area_for_the_US_July_2023_4764927935501855778.geojson")
us_msa = us_cbsa[us_cbsa['CBSATYPE'] == 'Metropolitan Statistical Area']
msa_siousfalls = us_msa[us_msa['CBSACODE']=='43620']
msa_siousfalls

,OBJECTID,CBSACODE,CBSANAME,CBSATYPE,ALAND,AWATER,geometry
794,795,43620,"Sioux Falls, SD-MN",Metropolitan Statistical Area,7919825565,28592302,"POLYGON ((-97.15559 43.08313, -97.15781 43.083..."


In [65]:
msa_siousfalls.bounds

,minx,miny,maxx,maxy
794,-97.609105,43.083131,-96.052304,43.849528


# Query from Foursquare

In [4]:
import os
import json
import base64
import numpy as np
import geopandas as gpd
from typing import Optional
from pyiceberg.expressions import And, GreaterThanOrEqual, LessThanOrEqual
from pyiceberg.catalog import load_catalog


def read_token(token_path: str) -> str:
    """
    Read and return the Foursquare API token from a local JSON file.

    The function supports two valid JSON formats:
    1) An object with a "token" key: {"token": "xxxxx"}
    2) A plain string containing the token: "xxxxx"

    Parameters
    ----------
    token_path : str
        Local file path to the JSON file containing the token.

    Returns
    -------
    str
        The token string extracted from the file.

    Raises
    ------
    ValueError
        If the JSON file format is invalid or does not contain a valid token.
    """
    with open(token_path, "r") as f:
        data = json.load(f)
    if isinstance(data, dict) and "token" in data:
        return data["token"]
    if isinstance(data, str):
        return data
    raise ValueError("Token JSON must be either a string or an object with a key 'token'.")


def json_str_tr(x):
    """
    Convert non-JSON-serializable Python objects (e.g., lists, dicts, arrays)
    into string representations that can be safely exported or visualized.

    Parameters
    ----------
    x : Any
        The input value to be converted.

    Returns
    -------
    str or None
        The JSON-compatible string representation of the input value.
    """
    if x is None:
        return None
    elif isinstance(x, (str, int, float, bool)):
        return str(x)
    elif isinstance(x, (list, tuple, set, np.ndarray)):
        return str(list(x))
    elif isinstance(x, dict):
        return str({k: json_str_tr(v) for k, v in x.items()})
    elif isinstance(x, (bytes, bytearray)):
        return base64.b64encode(x).decode("ascii")
    else:
        return str(x)


def query_foursquare(
    token_path: str,
    minLon: float,
    maxLon: float,
    minLat: float,
    maxLat: float,
    limit_size: Optional[int] = None,
    table_name: str = "datasets.places_os",
    uri: str = "https://catalog.h3-hub.foursquare.com/iceberg",
    warehouse: str = "places",
):
    """
    Query Points of Interest (POIs) from the Foursquare Iceberg Catalog
    within a given geographic bounding box and return results as a GeoDataFrame.

    This function connects to the Foursquare H3-Hub Iceberg dataset, applies a spatial filter
    (latitude/longitude bounds), retrieves the data as a Pandas DataFrame, converts it into
    a GeoDataFrame (EPSG:4326), and ensures all complex fields are JSON-serializable.

    Parameters
    ----------
    token_path : str
        Path to the local JSON file containing the Foursquare API token.
        The file should follow one of these formats:
        - {"token": "xxxxx"}
        - "xxxxx"
    minLon, maxLon, minLat, maxLat : float
        Geographic bounding box coordinates (in WGS84) for the query.
    limit_size : int, optional
        Maximum number of rows to return. If None (default), returns all available rows.
    table_name : str, default "datasets.places_os"
        Name of the Iceberg table to query.
    uri : str, default "https://catalog.h3-hub.foursquare.com/iceberg"
        Base URI of the Foursquare Iceberg REST catalog.
    warehouse : str, default "places"
        Name of the warehouse used by the Iceberg catalog.

    Returns
    -------
    geopandas.GeoDataFrame
        A GeoDataFrame containing POI records with WGS84 geometry.

    Examples
    --------
    Limit to 5000 rows:
    >>> gdf = query_foursquare(
    ...     token_path="~/foursquare_token.json",
    ...     minLon=-119.8694, maxLon=-119.85346,
    ...     minLat=34.40887, maxLat=34.41727,
    ...     limit_size=5000
    ... )

    Retrieve all available records (no limit):
    >>> gdf = query_foursquare(
    ...     token_path="~/foursquare_token.json",
    ...     minLon=-119.8694, maxLon=-119.85346,
    ...     minLat=34.40887, maxLat=34.41727
    ... )
    """
    # --- Load token ---
    token = read_token(os.path.expanduser(token_path))

    # --- Connect to the Iceberg catalog ---
    catalog = load_catalog(
        "default",
        **{
            "warehouse": warehouse,
            "uri": uri,
            "token": token,
            "header.content-type": "application/vnd.api+json",
            "rest-metrics-reporting-enabled": "false",
        },
    )

    # --- Load target table ---
    table = catalog.load_table(table_name)

    # --- Define spatial filter ---
    expr = And(
        And(GreaterThanOrEqual("longitude", minLon), LessThanOrEqual("longitude", maxLon)),
        And(GreaterThanOrEqual("latitude", minLat), LessThanOrEqual("latitude", maxLat)),
    )

    # --- Run query ---
    df = table.scan(row_filter=expr, limit=limit_size).to_pandas()

    # --- Convert to GeoDataFrame ---
    gdf = gpd.GeoDataFrame(
        df,
        geometry=gpd.points_from_xy(df["longitude"], df["latitude"]),
        crs="EPSG:4326",
    )

    # --- Clean up non-serializable columns ---
    for col in gdf.columns:
        if col != "geometry":
            if gdf[col].apply(lambda x: isinstance(x, (list, dict, np.ndarray, bytes, bytearray))).any():
                gdf[col] = gdf[col].apply(json_str_tr)

    return gdf


In [66]:
msa_siousfalls.bounds

,minx,miny,maxx,maxy
794,-97.609105,43.083131,-96.052304,43.849528


In [67]:
foursquare_places = query_foursquare(token_path="/Users/houpuli/Redlining Lab Dropbox/HOUPU LI/foursquare_token.json",
                       minLon=-97.609105, maxLon=-96.052304,
                       minLat=43.083131, maxLat=43.849528)

In [68]:
import ast

def safe_parse(x):
    if x is None or (isinstance(x, float)):
        return None
    try:
        result = ast.literal_eval(x)
        return result[0] if isinstance(result, list) and len(result) > 0 else None
    except:
        return x

foursquare_places["cat_str"] = foursquare_places["fsq_category_labels"].apply(safe_parse)
cats = foursquare_places["cat_str"].str.split(" > ", expand=True)
foursquare_places["cat_main"] = cats[0]
foursquare_places["cat_alt"] = cats[1]

foursquare_places["fsq_category_ids"] = foursquare_places["fsq_category_ids"].apply(lambda x: ", ".join(ast.literal_eval(x)) if isinstance(x, str) and x.startswith("[") else x)
foursquare_places["fsq_category_labels"] = foursquare_places["fsq_category_labels"].apply(lambda x: ", ".join(ast.literal_eval(x)) if isinstance(x, str) and x.startswith("[") else x)
foursquare_places

,fsq_place_id,name,latitude,longitude,address,locality,region,postcode,admin_region,post_town,...,fsq_category_ids,fsq_category_labels,placemaker_url,unresolved_flags,geom,bbox,geometry,cat_str,cat_main,cat_alt
0,62bde45e25ecb17924df322d,Exit 353,43.665276,-97.588120,None,Spencer,SD,57374,None,None,...,52f2ab2ebcbc57f1066b8b4c,Travel and Transportation > Road > Intersection,https://foursquare.com/placemakers/review-plac...,None,AAAAAAHAWGWjwLgwy0BF1SfDo1+P,"{'xmin': '-97.5881196783768', 'ymin': '43.6652...",POINT (-97.58812 43.66528),Travel and Transportation > Road > Intersection,Travel and Transportation,Road
1,4e022400aeb76b61098a364b,Subway,43.663025,-97.587297,25678 431st Ave,Spencer,SD,57374,None,None,...,"4bf58dd8d48988d1c5941735, 4bf58dd8d48988d16e94...",Dining and Drinking > Restaurant > Sandwich Sp...,https://foursquare.com/placemakers/review-plac...,None,AAAAAAHAWGWWSDnUAEBF1N3+xrmK,"{'xmin': '-97.58729749343183', 'ymin': '43.663...",POINT (-97.58730 43.66302),Dining and Drinking > Restaurant > Sandwich Spot,Dining and Drinking,Restaurant
2,4bd4d22a4e32d13a20febf80,Fuel Mart,43.663128,-97.587268,None,Emery,SD,57332,None,None,...,4bf58dd8d48988d113951735,Travel and Transportation > Fuel Station,https://foursquare.com/placemakers/review-plac...,None,AAAAAAHAWGWVywoM5EBF1OFfq9iO,"{'xmin': '-97.58726764661293', 'ymin': '43.663...",POINT (-97.58727 43.66313),Travel and Transportation > Fuel Station,Travel and Transportation,Fuel Station
3,4ff12392e4b0279cf31e3676,Spencer,43.664962,-97.578009,None,Spencer,SD,57374,None,None,...,4f2a25ac4b909258e854f55f,Landmarks and Outdoors > States and Municipali...,https://foursquare.com/placemakers/review-plac...,None,AAAAAAHAWGT+GOxWaUBF1R15h8DJ,"{'xmin': '-97.57800887183988', 'ymin': '43.664...",POINT (-97.57801 43.66496),Landmarks and Outdoors > States and Municipali...,Landmarks and Outdoors,States and Municipalities
4,4d4af7fd11a36ea814dd361c,Spencer AT&T Tower Site,43.664986,-97.537782,None,Spencer,SD,None,None,None,...,None,None,https://foursquare.com/placemakers/review-plac...,None,AAAAAAHAWGJrBunJkkBF1R5FxEzD,"{'xmin': '-97.5377824099617', 'ymin': '43.6649...",POINT (-97.53778 43.66499),None,None,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
22281,f1950d6f567645f9310c45fa,Darwin Kirschenman Farm,43.138999,-97.472663,43617 293rd St,Menno,SD,57045,None,None,...,4bf58dd8d48988d15b941735,Landmarks and Outdoors > Farm,https://foursquare.com/placemakers/review-plac...,None,AAAAAAHAWF5AGmkOj0BFkcq7j0J1,"{'xmin': '-97.47266254672489', 'ymin': '43.138...",POINT (-97.47266 43.13900),Landmarks and Outdoors > Farm,Landmarks and Outdoors,Farm
22282,1451b0852a254e0de2b03b16,Munkvold Land & Cattle Co.,43.124079,-97.454544,43723 294th St,Menno,SD,57045,None,None,...,None,None,https://foursquare.com/placemakers/review-plac...,None,AAAAAAHAWF0XPmGCVUBFj+HQ3kIK,"{'xmin': '-97.45454368135809', 'ymin': '43.124...",POINT (-97.45454 43.12408),None,None,None
22283,536d58ad498ee2f0d55b2102,Jamesville,43.126017,-97.438841,None,Jamesville,SD,None,None,None,...,50aa9e094b90af0d42d5de0d,Landmarks and Outdoors > States and Municipali...,https://foursquare.com/placemakers/review-plac...,None,AAAAAAHAWFwV9sxqUkBFkCFTRhId,"{'xmin': '-97.4388405796283', 'ymin': '43.1260...",POINT (-97.43884 43.12602),Landmarks and Outdoors > States and Municipali...,Landmarks and Outdoors,States and Municipalities
22284,8212d644987e483ff1e331a2,Jamesville Hutterian Brethren,43.101461,-97.480511,29568 436th Ave,Utica,SD,57067,None,None,...,63be6904847c3692a84b9b9a,Community and Government,https://foursquare.com/placemakers/review-plac...,None,AAAAAAHAWF7AsmUHdkBFjPysl0qm,"{'xmin': '-97.4805112825978', 'ymin': '43.1014...",POINT (-97.48051 43.10146),Community and Government,Community and Government,None


In [70]:
foursquare_places.to_file('/Users/houpuli/Redlining Lab Dropbox/HOUPU LI/POI research/Sioux_Falls_MSA/Sious_Falls_msa_poi/sioux_falls_fsq.geojson', driver="GeoJSON")

# Query by OSM

In [10]:
delineation_url = (
    "https://www2.census.gov/programs-surveys/metro-micro/"
    "geographies/reference-files/2020/delineation-files/list1_2020.xls"
)
df = pd.read_excel(delineation_url, header=2)
df.columns = [str(c).strip() for c in df.columns]
df["CBSA Code"] = df["CBSA Code"].astype(str).str.split(".").str[0].str.zfill(5)
houston_counties = df[df["CBSA Code"] == "26420"]
print(houston_counties[["State Name", "FIPS State Code", "County/County Equivalent"]].to_string())

    State Name  FIPS State Code County/County Equivalent
764      Texas             48.0            Austin County
765      Texas             48.0          Brazoria County
766      Texas             48.0          Chambers County
767      Texas             48.0         Fort Bend County
768      Texas             48.0         Galveston County
769      Texas             48.0            Harris County
770      Texas             48.0           Liberty County
771      Texas             48.0        Montgomery County
772      Texas             48.0            Waller County


In [71]:
import pandas as pd
from pyrosm import OSM

def extract_comprehensive_pois(pbf):
    """
    Extracts POIs from an OSM PBF file and cleanses the data for spatial analysis.
    """
    # Initialize OSM parser
    osm = OSM(pbf)
    
    # Define broad filter to capture maximum POI density
    custom_filter = {
        'amenity': True, 'shop': True, 'tourism': True, 'leisure': True, 
        'office': True, 'healthcare': True, 'religion': True, 'emergency': True,
        'historic': True, 'government': True, 'craft': True, 'public_transport': True,
    }
    
    # Load data
    pois = osm.get_pois(custom_filter=custom_filter)
    if pois is None: return None

    # Convert Polygons to Points and defragment DataFrame to fix PerformanceWarnings
    pois['geometry'] = pois.geometry.centroid
    pois = pois.copy()

    # Unified Category Logic
    cat_columns = [
        'amenity', 'shop', 'tourism', 'leisure', 'office', 'healthcare', 
        'religion', 'emergency', 'historic', 'government', 'craft', 'public_transport'
    ]
    existing_cats = [c for c in cat_columns if c in pois.columns]
    pois['cat'] = pois[existing_cats].bfill(axis=1).iloc[:, 0]
    
    # Address Reconstruction Function
    def build_address(row):
        num = str(row.get('addr:housenumber', '')).strip()
        street = str(row.get('addr:street', '')).strip()
        num = '' if num.lower() in ['nan', 'none'] else num
        street = '' if street.lower() in ['nan', 'none'] else street
        
        if num and street:
            return f"{num} {street}"
        elif street: 
            return street
        
        # Fallback to building name
        housename = str(row.get('addr:housename', '')).strip()
        if housename and housename.lower() not in ['nan', 'none']:
            return housename
        return "N/A"

    pois['address'] = pois.apply(build_address, axis=1)
    
    # Filter final schema
    output_cols = ['id','timestamp','name', 'cat', 'address', 'tags', 'visible', 'version', 'geometry']
    return pois[output_cols]

gdf = extract_comprehensive_pois("/Users/houpuli/Downloads/south-dakota-260426.osm.pbf")
# gdf_in = extract_comprehensive_pois("/Users/houpuli/Downloads/indiana-260330.osm.pbf")
# gdf_wi = extract_comprehensive_pois("/Users/houpuli/Downloads/wisconsin-260330.osm.pbf")
# gdf_nj = extract_comprehensive_pois("/Users/houpuli/Downloads/new-jersey-260218.osm.pbf")

/var/folders/s5/1g0kvw1x37z1n8xgxcp8f3lh0000gn/T/ipykernel_59733/2024444290.py:23: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  pois['geometry'] = pois.geometry.centroid


In [72]:
msa_siousfalls = us_msa[us_msa['CBSACODE']=='43620']
# msa_chicago_osm = pd.concat([gdf_il, gdf_in, gdf_wi], ignore_index=True)
msa_siousfalls_osm = gpd.sjoin(gdf, msa_siousfalls, how='inner', predicate='within').reset_index(drop=True)
msa_siousfalls_osm = msa_siousfalls_osm.drop(columns=['index_right','OBJECTID','CBSACODE','CBSANAME','CBSATYPE','ALAND','AWATER'])
msa_siousfalls_osm['id'] = msa_siousfalls_osm['id'].astype(str)
msa_siousfalls_osm['timestamp'] = msa_siousfalls_osm['timestamp'].astype(str)
msa_siousfalls_osm = msa_siousfalls_osm.drop_duplicates(subset=['id'], keep='first').reset_index(drop=True)

In [75]:
def normalize_nulls(df):
    NULL_LIKE = ["none", "null", "nan", "n/a", "na", "", " ", "   "]
    df = df.copy()

    geom = None
    if isinstance(df, gpd.GeoDataFrame):
        geom = df.geometry
        geom_name = df.geometry.name


    cols = df.columns if geom is None else df.columns.drop(geom_name)
    df[cols] = df[cols].replace(r"^\s*$", pd.NA, regex=True)
    df[cols] = df[cols].replace(
        [x.upper() for x in NULL_LIKE] +
        [x.lower() for x in NULL_LIKE] +
        [x.capitalize() for x in NULL_LIKE],
        pd.NA
    )

    df[cols] = df[cols].replace({None: pd.NA})
    df[cols] = df[cols].where(pd.notna(df[cols]), pd.NA)

    if geom is not None:
        df = gpd.GeoDataFrame(df, geometry=geom, crs=geom.crs)

    return df

msa_siousfalls_osm = normalize_nulls(msa_siousfalls_osm)

In [76]:
import json

def extract_from_tags(tag_str, priority_keys):
    if pd.isna(tag_str) or not tag_str or tag_str == 'None':
        return None
    try:
        tags_dict = json.loads(tag_str)
        for key in priority_keys:
            if key in tags_dict:
                return tags_dict[key]
    except:
        return None
    return None

target_keys = [
    'amenity', 'shop', 'tourism', 'leisure', 'healthcare', 
    'office', 'religion', 'emergency', 'historic', 
    'government', 'craft', 'public_transport', 'building'
]

mask = msa_siousfalls_osm['cat'].isna()
msa_siousfalls_osm.loc[mask, 'cat'] = msa_siousfalls_osm.loc[mask, 'tags'].apply(lambda x: extract_from_tags(x, target_keys))

In [77]:
msa_siousfalls_osm.to_file('/Users/houpuli/Redlining Lab Dropbox/HOUPU LI/POI research/Sioux_Falls_MSA/Sious_Falls_msa_poi/siousfalls_osm.geojson')

# Query by Safegraph

In [78]:
import geopandas as gpd
import pandas as pd
from glob import glob

folder = "/Users/houpuli/Redlining Lab Dropbox/HOUPU LI/POI research/Sioux_Falls_MSA/Sious_Falls_msa_poi/safegraph_raw"
files = glob(folder + "/*.snappy.parquet")

# print(len(files))  

dfs = []

# for f in files:
#     print("reading:", f)
#     dfs.append(pd.read_csv(f))
    
dfs = []
for f in files:
    print("reading:", f)
    dfs.append(pd.read_parquet(f))

sd = pd.concat(dfs, ignore_index=True)
sd = sd[['PLACEKEY','PARENT_PLACEKEY','LOCATION_NAME', 'TRACKING_CLOSED_SINCE', 'LATITUDE','LONGITUDE','TOP_CATEGORY','SUB_CATEGORY','CATEGORY_TAGS','NAICS_CODE','STREET_ADDRESS']]
sd = gpd.GeoDataFrame(sd, geometry=gpd.points_from_xy(sd["LONGITUDE"], sd["LATITUDE"]), crs="EPSG:4326")
sd = sd.drop(columns=['LATITUDE','LONGITUDE'])

msa_siousfalls = msa_siousfalls.to_crs(sd.crs)
siousfalls_sf = gpd.sjoin(sd, msa_siousfalls, how='inner',predicate="within").reset_index(drop=True)
siousfalls_sf = siousfalls_sf.drop(columns=['index_right','OBJECTID','CBSACODE','CBSANAME','CBSATYPE','ALAND','AWATER'])
siousfalls_sf["TRACKING_CLOSED_SINCE"] = siousfalls_sf["TRACKING_CLOSED_SINCE"].astype(str).replace("None", None).replace("nan", None)

reading: /Users/houpuli/Redlining Lab Dropbox/HOUPU LI/POI research/Sioux_Falls_MSA/Sious_Falls_msa_poi/safegraph_raw/global-places-poi-geometry_0_6_0.snappy.parquet
reading: /Users/houpuli/Redlining Lab Dropbox/HOUPU LI/POI research/Sioux_Falls_MSA/Sious_Falls_msa_poi/safegraph_raw/global-places-poi-geometry_2_5_0.snappy.parquet
reading: /Users/houpuli/Redlining Lab Dropbox/HOUPU LI/POI research/Sioux_Falls_MSA/Sious_Falls_msa_poi/safegraph_raw/global-places-poi-geometry_1_1_0.snappy.parquet
reading: /Users/houpuli/Redlining Lab Dropbox/HOUPU LI/POI research/Sioux_Falls_MSA/Sious_Falls_msa_poi/safegraph_raw/global-places-poi-geometry_3_2_0.snappy.parquet
reading: /Users/houpuli/Redlining Lab Dropbox/HOUPU LI/POI research/Sioux_Falls_MSA/Sious_Falls_msa_poi/safegraph_raw/global-places-poi-geometry_3_6_0.snappy.parquet
reading: /Users/houpuli/Redlining Lab Dropbox/HOUPU LI/POI research/Sioux_Falls_MSA/Sious_Falls_msa_poi/safegraph_raw/global-places-poi-geometry_1_5_0.snappy.parquet
read

In [80]:
siousfalls_sf.to_parquet('/Users/houpuli/Redlining Lab Dropbox/HOUPU LI/POI research/Sioux_Falls_MSA/Sious_Falls_msa_poi/siousfalls_sf.parquet')

# Query by Overture

In [81]:
import json
import pandas as pd

def safe_json_load(x):
    try:
        return json.loads(x) if isinstance(x, str) else x
    except:
        return None

def extract_categories(x):
    data = safe_json_load(x)
    if isinstance(data, dict):
        primary = data.get('primary')
        alternate_list = data.get('alternate', [])
        alternate_str = ", ".join(alternate_list) if alternate_list else None
        return primary, alternate_str
    return None, None

def extract_name(x):
    data = safe_json_load(x)
    if isinstance(data, dict):
        return data.get('primary')
    return None

def extract_address(x):
    data = safe_json_load(x)
    if isinstance(data, list) and len(data) > 0:
        return data[0].get('freeform')
    return None

def extract_sources(x):
    data = safe_json_load(x)
    if isinstance(data, list) and len(data) > 0:
        first = data[0] 
        return first.get('dataset'), first.get('record_id'), first.get('update_time')
    return None, None, None


siousfalls_ove = pd.read_csv('/Users/houpuli/Redlining Lab Dropbox/HOUPU LI/POI research/Sioux_Falls_MSA/sious_falls_ove.csv')

siousfalls_ove[['cat_main', 'cat_alt']] = siousfalls_ove['categories'].apply(
    lambda x: pd.Series(extract_categories(x))
)

siousfalls_ove['name'] = siousfalls_ove['names'].apply(extract_name)
siousfalls_ove['address'] = siousfalls_ove['addresses'].apply(extract_address)
siousfalls_ove[['src_dataset', 'src_record_id', 'src_update_time']] = siousfalls_ove['sources'].apply(lambda x: pd.Series(extract_sources(x)))


siousfalls_ove['WKB'] = siousfalls_ove['geometry'].apply(lambda x: x[2:])
geometry = gpd.GeoSeries.from_wkb(siousfalls_ove['WKB'], crs=4326)

siousfalls_ove = gpd.GeoDataFrame(siousfalls_ove, geometry=geometry)
siousfalls_ove = siousfalls_ove[['id','name','address','cat_main','cat_alt','confidence','operating_status','version','src_dataset','src_update_time','geometry']]
siousfalls_ove

,id,name,address,cat_main,cat_alt,confidence,operating_status,version,src_dataset,src_update_time,geometry
0,461a4c39-39d8-4353-ad90-c913f12d6e99,Menno Pioneer Opry,611 N 5th St,music_venue,None,0.805395,open,7,meta,2026-04-06T00:00:00.000Z,POINT (-97.57595 43.24599)
1,996e8974-86a3-4ab6-aa5d-c7e22d99973f,Menno Public School,None,public_school,"middle_school, school",0.953197,open,6,meta,2026-04-06T00:00:00.000Z,POINT (-97.57805 43.24407)
2,422e38b0-1f16-46a1-911f-5fb0fe94c851,Menno Food Market,234 N 5th St,grocery_store,None,0.770000,open,7,Foursquare,2025-08-22T00:00:00.000,POINT (-97.57792 43.24323)
3,5fdf64ee-17de-4189-b204-87e57ce7831b,Menno Food Market,234 431st Ave,grocery_store,"supermarket, restaurant",0.368745,open,1,meta,2026-04-06T00:00:00.000Z,POINT (-97.57797 43.24283)
4,bd0fb9cc-fb94-4618-92a6-cf2444ba87df,Hutchinson County Highway Shop,411 N Access Rd,public_and_government_association,"public_service_and_government, central_governm...",0.783082,open,1,meta,2026-04-06T00:00:00.000Z,POINT (-97.57121 43.24250)
...,...,...,...,...,...,...,...,...,...,...,...
16995,7405763f-6968-40a6-b9d1-269b325748e7,Lebanon Christian Reformed Church,2142 380th St,church_cathedral,religious_organization,0.980608,open,6,meta,2026-04-06T00:00:00.000Z,POINT (-96.35782 43.09875)
16996,457e0266-f1ad-431b-a519-6384408833ee,Countryside Ag Service,2156 390th St,agricultural_service,"agriculture, agricultural_cooperatives",0.962733,open,9,meta,2026-04-06T00:00:00.000Z,POINT (-96.36064 43.08440)
16997,31a27745-4af0-4eb6-9104-24255f68fbdd,Doummar Sons,Near Amwaj Center,None,None,0.233295,open,1,meta,2026-04-06T00:00:00.000Z,POINT (-96.32985 43.08419)
16998,e2803354-163f-4219-a346-46a2aecf7098,Rosalinda,"hazmieh center 976, floor 4",childrens_clothing_store,None,0.113548,open,4,meta,2026-04-06T00:00:00.000Z,POINT (-96.32985 43.08419)


In [82]:
siousfalls_ove.to_file('/Users/houpuli/Redlining Lab Dropbox/HOUPU LI/POI research/Sioux_Falls_MSA/Sious_Falls_msa_poi/siousfalls_ove.geojson', driver="GeoJSON")

# Query by Google Place API

In [84]:
import math
import time
import requests
import json
import warnings
from typing import Optional, Iterable, List, Dict, Any, Tuple

import numpy as np
import pandas as pd
import geopandas as gpd
from shapely.geometry import box, Point
import matplotlib.pyplot as plt
from matplotlib.patches import Patch

def read_token(token_path: str) -> str:
    """
    Read and return the token from a local JSON file.

    The function supports two valid JSON formats:
    1) An object with a "token" key: {"token": "xxxxx"}
    2) A plain string containing the token: "xxxxx"

    Parameters
    ----------
    token_path : str
        Local file path to the JSON file containing the token.

    Returns
    -------
    str
        The token string extracted from the file.

    Raises
    ------
    ValueError
        If the JSON file format is invalid or does not contain a valid token.
    """
    with open(token_path, "r") as f:
        data = json.load(f)
    if isinstance(data, dict) and "token" in data:
        return data["token"]
    if isinstance(data, str):
        return data
    raise ValueError("Token JSON must be either a string or an object with a key 'token'.")


def circle_center(
    min_lon: float,
    max_lon: float,
    min_lat: float,
    max_lat: float,
    R: float,
    base: gpd.GeoDataFrame,
) -> gpd.GeoDataFrame:
    """
    Generate a grid of circle polygons inside a geographic bounding box.
    Returns circles in EPSG:4326 (lon/lat).

    If `base` is provided, return only circles that intersect `base`
    (via spatial join, predicate='intersects').

    Parameters
    ----------
    min_lon, max_lon, min_lat, max_lat : float
        Bounding box in EPSG:4326.
    R : float
        Grid half-spacing in meters (in EPSG:3857). Spacing is 2R.
        Each circle radius is R * sqrt(2) in meters (in EPSG:3857).
    base : geopandas.GeoDataFrame | None
        A GeoDataFrame used to filter circles by intersection.
        Will be reprojected to EPSG:4326 if needed.

    Returns
    -------
    geopandas.GeoDataFrame
        Circles (filtered if base is provided), in EPSG:4326.
    """
    # -----------------------------
    # 0) Basic input validation
    # -----------------------------
    if not (isinstance(min_lon, (int, float)) and isinstance(max_lon, (int, float))
            and isinstance(min_lat, (int, float)) and isinstance(max_lat, (int, float))):
        raise TypeError("min_lon, max_lon, min_lat, max_lat must be numeric (float or int).")
    if not (max_lon > min_lon and max_lat > min_lat):
        raise ValueError("max_lon must be > min_lon and max_lat must be > min_lat.")
    if not (isinstance(R, (int, float)) and R > 0):
        raise ValueError("R must be a positive number (meters).")

    if base is not None:
        if not isinstance(base, gpd.GeoDataFrame):
            raise TypeError("base must be a GeoDataFrame or None.")
        if base.geometry is None:
            raise ValueError("base must have a valid geometry column.")
        if base.crs is None:
            raise ValueError("base.crs is None. Please set a CRS on base before using it.")

    # -----------------------------
    # 1) Build bbox in EPSG:4326
    # -----------------------------
    poly_4326 = box(min_lon, min_lat, max_lon, max_lat)
    gdf_bbox_4326 = gpd.GeoDataFrame({"name": ["bbox_4326"]}, geometry=[poly_4326], crs="EPSG:4326")

    # -----------------------------
    # 2) Reproject bbox to EPSG:3857 (meters)
    # -----------------------------
    gdf_bbox_3857 = gdf_bbox_4326.to_crs(epsg=3857)
    minx, miny, maxx, maxy = gdf_bbox_3857.total_bounds

    # -----------------------------
    # 3) Grid centers inside bbox
    # -----------------------------
    xs = np.arange(minx + R, maxx + 1e-9, 2 * R)
    ys = np.arange(miny + R, maxy + 1e-9, 2 * R)

    if xs.size == 0 or ys.size == 0:
        warnings.warn("⚠️ No grid centers generated — check if R is too large or bbox too small.")
        centers_xy = np.empty((0, 2))
    else:
        xx, yy = np.meshgrid(xs, ys)
        centers_xy = np.column_stack([xx.ravel(), yy.ravel()])

    # -----------------------------
    # 4) Build circles in 3857 -> back to 4326
    # -----------------------------
    radius_m = R
    circles_3857 = [Point(cx, cy).buffer(radius_m, resolution=64) for (cx, cy) in centers_xy]
    circles = gpd.GeoDataFrame(geometry=circles_3857, crs="EPSG:3857").to_crs(epsg=4326)

    # -----------------------------
    # 5) Optional: filter circles by intersection with base
    # -----------------------------
    base_4326 = base.to_crs("EPSG:4326")
    circles_4326 = (
        gpd.sjoin(circles, base_4326, how="inner", predicate="intersects")
            .reset_index(drop=True)
    )
    circles_4326 = circles_4326[circles.columns]
    circles_m = circles_4326.to_crs(epsg=3857)
    centroids_m = circles_m.geometry.centroid
    centroids_ll = centroids_m.to_crs(epsg=4326)
    circles_4326["center_lon"] = centroids_ll.x
    circles_4326["center_lat"] = centroids_ll.y
    
    return circles_4326

def places_nearby_grid(
    circles: gpd.GeoDataFrame,
    *,
    token: str,
    R: float,
    place_types: List[str],
    field_mask: Iterable[str],
    max_result_count: int = 20,
    sleep_sec: float = 0.1,
) -> gpd.GeoDataFrame:
    """
    Query Google Places Nearby Search for each circle center.

    Parameters
    ----------
    circles : GeoDataFrame
        Must contain columns ['center_lon', 'center_lat'] in EPSG:4326.
    token : str
        Google Places API key.
    R : float
        Search radius in meters (should match circle_center R).
    place_types : list[str]
        includedPrimaryTypes sent to Google Places.
    field_mask : iterable[str]
        Fields requested via X-Goog-FieldMask.
    max_result_count : int, default 20
        Max results per query (Google cap).
    sleep_sec : float, default 0.1
        Sleep time between requests to avoid rate limiting.

    Returns
    -------
    GeoDataFrame
        POIs returned by Google Places Nearby Search (EPSG:4326).
    """

    url = "https://places.googleapis.com/v1/places:searchNearby"

    headers = {
        "Content-Type": "application/json",
        "X-Goog-Api-Key": token,
        "X-Goog-FieldMask": ",".join(field_mask),
    }

    rows: List[Dict[str, Any]] = []

    for circle_id, row in circles.iterrows():
        lat = row["center_lat"]
        lon = row["center_lon"]

        payload = {
            "locationRestriction": {
                "circle": {
                    "center": {
                        "latitude": float(lat),
                        "longitude": float(lon),
                    },
                    "radius": float(R),
                }
            },
            "includedPrimaryTypes": place_types,
            "maxResultCount": int(max_result_count),
        }

        resp = requests.post(url, headers=headers, json=payload, timeout=30)
        data = resp.json()

        places = data.get("places", [])

        for p in places:
            rows.append({
                "circle_id": circle_id,
                "id": p.get("id"),
                "name": p.get("displayName", {}).get("text"),
                "address": p.get("formattedAddress"),
                "primary_type": p.get("primaryType"),
                "lat": p.get("location", {}).get("latitude"),
                "lon": p.get("location", {}).get("longitude"),
                "business_status": p.get("businessStatus"),
            })

        time.sleep(sleep_sec)

    df = pd.DataFrame(rows)
    gdf = gpd.GeoDataFrame(
        df,
        geometry=gpd.points_from_xy(df["lon"], df["lat"]),
        crs="EPSG:4326",
    )

    return gdf

In [85]:
min_lon, max_lon = 	-97.609105, -96.052304
min_lat, max_lat = 43.083131, 43.849528

token = read_token("/Users/houpuli/Redlining Lab Dropbox/HOUPU LI/googleplace_token.json")
R = 5000  # meters

circles = circle_center(
    min_lon=min_lon,
    max_lon=max_lon,
    min_lat=min_lat,
    max_lat=max_lat,
    R=R,
    base=msa_siousfalls
)
circles

,geometry,center_lon,center_lat
0,"POLYGON ((-97.15995 43.11593, -97.15996 43.115...",-97.204863,43.115927
1,"POLYGON ((-97.07012 43.11593, -97.07013 43.115...",-97.115032,43.115927
2,"POLYGON ((-96.98028 43.11593, -96.98030 43.115...",-97.025200,43.115927
3,"POLYGON ((-96.89045 43.11593, -96.89047 43.115...",-96.935369,43.115927
4,"POLYGON ((-96.80062 43.11593, -96.80063 43.115...",-96.845537,43.115927
...,...,...,...
161,"POLYGON ((-96.44130 43.83299, -96.44131 43.832...",-96.486211,43.832993
162,"POLYGON ((-96.35146 43.83299, -96.35148 43.832...",-96.396379,43.832993
163,"POLYGON ((-96.26163 43.83299, -96.26165 43.832...",-96.306548,43.832993
164,"POLYGON ((-96.17180 43.83299, -96.17181 43.832...",-96.216716,43.832993


## 1.shop

In [86]:
field_mask = [
    "places.id",
    "places.displayName",
    "places.formattedAddress",
    "places.location",
    "places.primaryType",
    "places.businessStatus",
]

place_types = [
    "asian_grocery_store",
    "auto_parts_store",
    "bicycle_store",
    "book_store",
    "building_materials_store",
    "butcher_shop",
    "cell_phone_store",
    "clothing_store",
    "convenience_store",
    "cosmetics_store",
    "department_store",
    "discount_store",
    "discount_supermarket",
    "electronics_store",
    "farmers_market",
    "flea_market",
    "food_store",
    "furniture_store",
    "garden_center",
    "general_store",
    "gift_shop",
    "grocery_store",
    "hardware_store",
    "health_food_store",
    "home_goods_store",
    "home_improvement_store",
    "hypermarket",
    "jewelry_store",
    "liquor_store",
    "market",
    "pet_store",
    "shoe_store",
    "shopping_mall",
    "sporting_goods_store",
    "sportswear_store",
    "store",
    "supermarket",
    "tea_store",
    "thrift_store",
    "toy_store",
    "warehouse_store",
    "wholesaler",
    "womens_clothing_store",
]

google_shop_5000 = places_nearby_grid(
    circles,
    token=token,
    R=R,
    place_types=place_types,
    field_mask=field_mask,
    sleep_sec=0.1,
)

google_shop_5000

,circle_id,id,name,address,primary_type,lat,lon,business_status,geometry
0,0,ChIJi2ol6LYcj4cRGI6E-GtRKcs,Dick's Country Butcher Shop,"45151 295th St, Irene, SD 57037, USA",butcher_shop,43.110250,-97.170549,OPERATIONAL,POINT (-97.17055 43.11025)
1,1,ChIJi2ol6LYcj4cRGI6E-GtRKcs,Dick's Country Butcher Shop,"45151 295th St, Irene, SD 57037, USA",butcher_shop,43.110250,-97.170549,OPERATIONAL,POINT (-97.17055 43.11025)
2,1,ChIJV_j2KOQCj4cRkDgKCZChuC8,Whimsies & Preheim Studios,"45362 SD-46, Irene, SD 57037, USA",store,43.083924,-97.129588,CLOSED_TEMPORARILY,POINT (-97.12959 43.08392)
3,1,ChIJN219-y4Dj4cRh2HpOJrR9sg,Heine antiques,"140 E Clay St, Irene, SD 57037, USA",store,43.082433,-97.155881,OPERATIONAL,POINT (-97.15588 43.08243)
4,3,ChIJe8QT1dT6jocRTOr7yYRmlrg,Zebra King Donuts,"516 Broadway St, Centerville, SD 57014, USA",donut_shop,43.117844,-96.960861,OPERATIONAL,POINT (-96.96086 43.11784)
...,...,...,...,...,...,...,...,...,...
796,165,ChIJU3M_ECHri4cR-E-AQnD9b0M,GuidePoint Pharmacy #111,"735 Main St, Edgerton, MN 56128, USA",pharmacy,43.874160,-96.129014,OPERATIONAL,POINT (-96.12901 43.87416)
797,165,ChIJzZH9yqeTi4cRzoOiKv849mE,NAPA Auto Parts - Auto Parts Edgerton,"801 Main St, Edgerton, MN 56128, USA",auto_parts_store,43.873466,-96.129049,OPERATIONAL,POINT (-96.12905 43.87347)
798,165,ChIJzZH9yqeTi4cRVBtTceUyWA4,Katie's Closet & More,"829 Main St, Edgerton, MN 56128, USA",store,43.873042,-96.128988,OPERATIONAL,POINT (-96.12899 43.87304)
799,165,ChIJrxQz-nfui4cRbOp3c204JHI,Remembered Moments,"2217 160th Ave, Edgerton, MN 56128, USA",store,43.806534,-96.153144,OPERATIONAL,POINT (-96.15314 43.80653)


In [87]:
google_shop_5000.to_file('/Users/houpuli/Redlining Lab Dropbox/HOUPU LI/POI research/Sioux_Falls_MSA/Sious_Falls_google_place_5000_bycat/google_shop_5000.geojson', driver="GeoJSON")

In [ ]:
# google_shop_5000 = gpd.read_file('/Users/houpuli/Redlining Lab Dropbox/HOUPU LI/POI research/LA MSA/la_google_place_5000_bycat/google_shop_5000.geojson', driver="GeoJSON")

## 2.automotive

In [88]:
field_mask = [
    "places.id",
    "places.displayName",
    "places.formattedAddress",
    "places.location",
    "places.primaryType",
    "places.businessStatus",
]

place_types = [
    "car_dealer",
    "car_rental",
    "car_repair",
    "car_wash",
    "ebike_charging_station",
    "electric_vehicle_charging_station",
    "gas_station",
    "parking",
    "parking_garage",
    "parking_lot",
    "rest_stop",
    "tire_shop",
    "truck_dealer",
]

google_automotive_5000 = places_nearby_grid(
    circles,
    token=token,
    R=R,
    place_types=place_types,
    field_mask=field_mask,
    sleep_sec=0.1,
)

google_automotive_5000

,circle_id,id,name,address,primary_type,lat,lon,business_status,geometry
0,3,ChIJKcdFrdX6jocRLCNhV3PTaMk,Pump N Stuff Centerville,"831 Broadway St, Centerville, SD 57014, USA",gas_station,43.114539,-96.961369,OPERATIONAL,POINT (-96.96137 43.11454)
1,3,ChIJE0li1tr7jocR3jexlnq3qlk,Mr. G's Tires,"911 Garfield St, Centerville, SD 57014, USA",tire_shop,43.115963,-96.961639,OPERATIONAL,POINT (-96.96164 43.11596)
2,3,ChIJyYiFINX6jocRZzmUvTNhjTM,NAPA Auto Parts - Centerville Farm & Auto Supply,"628 Broadway St, Centerville, SD 57014, USA",auto_parts_store,43.116461,-96.960883,OPERATIONAL,POINT (-96.96088 43.11646)
3,3,ChIJX5Hps9X6jocROrK1BoRFQGM,Phillips 66,"831 Broadway St, Centerville, SD 57014, USA",gas_station,43.115638,-96.961478,OPERATIONAL,POINT (-96.96148 43.11564)
4,3,ChIJl6N7YCnljocR4dpUp648Sx4,DK Auto,"800 North St, Centerville, SD 57014, USA",car_dealer,43.123161,-96.959936,OPERATIONAL,POINT (-96.95994 43.12316)
...,...,...,...,...,...,...,...,...,...
682,165,ChIJPZEmoJXri4cRy6IOuOGxMAI,Arv's Repair,"101 Maple St, Edgerton, MN 56128, USA",car_repair,43.873544,-96.129098,OPERATIONAL,POINT (-96.12910 43.87354)
683,165,ChIJz9w8UpTri4cRVXq-uMQL7rc,Chevrolet Parts,"930 Main St, Edgerton, MN 56128, USA",auto_parts_store,43.872242,-96.128304,OPERATIONAL,POINT (-96.12830 43.87224)
684,165,ChIJPZEmoJXri4cRGi4q9VhVU4I,Dragstra Auto Body,"111 Maple St, Edgerton, MN 56128, USA",car_repair,43.873484,-96.129558,OPERATIONAL,POINT (-96.12956 43.87348)
685,165,ChIJq3jZvJXri4cRUipqEwSKXik,Ppg Collision Repair Center,"101 Maple St, Edgerton, MN 56128, USA",car_repair,43.873535,-96.128128,OPERATIONAL,POINT (-96.12813 43.87354)


In [89]:
google_automotive_5000.to_file('/Users/houpuli/Redlining Lab Dropbox/HOUPU LI/POI research/Sioux_Falls_MSA/Sious_Falls_google_place_5000_bycat/google_automotive_5000.geojson', driver="GeoJSON")

In [ ]:
# google_automotive_5000 = gpd.read_file('/Users/houpuli/Downloads/google_place_5000_bycat/google_automotive_5000.geojson', driver="GeoJSON")

## 3.business

In [90]:
field_mask = [
    "places.id",
    "places.displayName",
    "places.formattedAddress",
    "places.location",
    "places.primaryType",
    "places.businessStatus",
]

place_types = [
    "business_center",
    "corporate_office",
    "coworking_space",
    "farm",
    "manufacturer",
    "ranch",
    "supplier",
    "television_studio",
]

google_business_5000 = places_nearby_grid(
    circles,
    token=token,
    R=R,
    place_types=place_types,
    field_mask=field_mask,
    sleep_sec=0.1,
)

google_business_5000

,circle_id,id,name,address,primary_type,lat,lon,business_status,geometry
0,0,ChIJi2ol6LYcj4cRGI6E-GtRKcs,Dick's Country Butcher Shop,"45151 295th St, Irene, SD 57037, USA",butcher_shop,43.110250,-97.170549,OPERATIONAL,POINT (-97.17055 43.11025)
1,0,ChIJWfM0amIbj4cRvFWqdOnl_QY,Olsen Yorkshires,"29369 451st Ave, Irene, SD 57037, USA",ranch,43.130928,-97.183382,OPERATIONAL,POINT (-97.18338 43.13093)
2,0,ChIJhzg4u2sDj4cR0XJ4fxKyIxk,Davis IronWorks,"420 W Main St, Irene, SD 57037, USA",manufacturer,43.083571,-97.162914,OPERATIONAL,POINT (-97.16291 43.08357)
3,1,ChIJi2ol6LYcj4cRGI6E-GtRKcs,Dick's Country Butcher Shop,"45151 295th St, Irene, SD 57037, USA",butcher_shop,43.110250,-97.170549,OPERATIONAL,POINT (-97.17055 43.11025)
4,2,ChIJxQ0cCPz7jocRI1oGpaIJTDw,Centerville Manufacturing,"1280 Garfield St, Centerville, SD 57014, USA",manufacturer,43.116064,-96.966412,OPERATIONAL,POINT (-96.96641 43.11606)
...,...,...,...,...,...,...,...,...,...
721,165,ChIJM5Gs1JPri4cRs3cJyorQHmE,De Kam Seed & Fertilizer LLC,"1300 Mechanic St #4408, Edgerton, MN 56128, USA",supplier,43.868262,-96.126621,OPERATIONAL,POINT (-96.12662 43.86826)
722,165,ChIJGTR7Tvvti4cRY3_NMOa_R38,Nykamp Cattle,"21 175th Ave, Edgerton, MN 56128, USA",farm,43.851419,-96.114776,OPERATIONAL,POINT (-96.11478 43.85142)
723,165,ChIJp74YMQDti4cRG-qppJ3F-SQ,Bouw Angus,"1649 231st St, Edgerton, MN 56128, USA",farm,43.820291,-96.143218,OPERATIONAL,POINT (-96.14322 43.82029)
724,165,ChIJbQd3UOrri4cRtqiEY9VpWPQ,Sioux Dairy Equipment Inc,"1006 4th Ave N S, Edgerton, MN 56128, USA",ranch,43.872805,-96.134597,OPERATIONAL,POINT (-96.13460 43.87281)


In [91]:
google_business_5000.to_file('/Users/houpuli/Redlining Lab Dropbox/HOUPU LI/POI research/Sioux_Falls_MSA/Sious_Falls_google_place_5000_bycat/google_business_5000.geojson', driver="GeoJSON")

In [ ]:
# google_business_5000 = gpd.read_file('/Users/houpuli/Downloads/google_place_5000_bycat/google_business_5000.geojson', driver="GeoJSON")

## 4.culture

In [92]:
field_mask = [
    "places.id",
    "places.displayName",
    "places.formattedAddress",
    "places.location",
    "places.primaryType",
    "places.businessStatus",
]

place_types = [
    "art_gallery",
    "art_museum",
    "art_studio",
    "auditorium",
    "castle",
    "cultural_landmark",
    "fountain",
    "historical_place",
    "history_museum",
    "monument",
    "museum",
    "performing_arts_theater",
    "sculpture",
]

google_culture_5000 = places_nearby_grid(
    circles,
    token=token,
    R=R,
    place_types=place_types,
    field_mask=field_mask,
    sleep_sec=0.1,
)

google_culture_5000

,circle_id,id,name,address,primary_type,lat,lon,business_status,geometry
0,1,ChIJZYKWSQADj4cR3ktaQ3tBMKA,Irene Water Tower,"401 N Clark Ave, Irene, SD 57037, USA",historical_landmark,43.083337,-97.152966,OPERATIONAL,POINT (-97.15297 43.08334)
1,3,ChIJAwU-OOL7jocRecU8S8_2bkc,Centerville Museum & Art Gallery,"616 Broadway St, Centerville, SD 57014, USA",museum,43.116751,-96.960807,OPERATIONAL,POINT (-96.96081 43.11675)
2,8,ChIJMU2Ka5iDjocRVq17th0fNkQ,Mouth of the Rock River,"Hawarden, IA 51023, USA",historical_landmark,43.082302,-96.453853,OPERATIONAL,POINT (-96.45385 43.08230)
3,13,ChIJzfii3pvjjocRMORyUkjRTkI,Daneville Heritage Museum,"200 1/2, 200 W Park Ave, Viborg, SD 57070, USA",museum,43.170520,-97.083167,OPERATIONAL,POINT (-97.08317 43.17052)
4,13,ChIJxfNU0X_jjocR8ajZsKjBqZg,Scandinavian Coop at The Creamery,"105 W Park Ave, Viborg, SD 57070, USA",art_gallery,43.170115,-97.082410,OPERATIONAL,POINT (-97.08241 43.17011)
...,...,...,...,...,...,...,...,...,...
110,140,ChIJV0xQ8wk3iYcR448c_5mk8GU,Lars Berven Homestead Bldg Site,"25165 469th Ave, Baltic, SD 57003, USA",historical_landmark,43.737534,-96.832103,CLOSED_TEMPORARILY,POINT (-96.83210 43.73753)
111,146,ChIJdzJ5I-_6i4cRu-5QleveXpM,"Touch The Sky Prairie, North Parking","181st St, Luverne, MN 56156, USA",museum,43.747328,-96.267316,OPERATIONAL,POINT (-96.26732 43.74733)
112,147,ChIJdzJ5I-_6i4cRu-5QleveXpM,"Touch The Sky Prairie, North Parking","181st St, Luverne, MN 56156, USA",museum,43.747328,-96.267316,OPERATIONAL,POINT (-96.26732 43.74733)
113,158,ChIJ7Zfoq9RAiYcR6FCCSUuh38I,Dell Rapids Society For History,"407 E 4th St, Dell Rapids, SD 57022, USA",museum,43.823094,-96.711501,OPERATIONAL,POINT (-96.71150 43.82309)


In [93]:
google_culture_5000.to_file('/Users/houpuli/Redlining Lab Dropbox/HOUPU LI/POI research/Sioux_Falls_MSA/Sious_Falls_google_place_5000_bycat/google_culture_5000.geojson', driver="GeoJSON")

In [ ]:
# google_culture_5000 = gpd.read_file('/Users/houpuli/Downloads/google_place_5000_bycat/google_culture_5000.geojson', driver="GeoJSON")

## 5.education

In [94]:
field_mask = [
    "places.id",
    "places.displayName",
    "places.formattedAddress",
    "places.location",
    "places.primaryType",
    "places.businessStatus",
]

place_types = [
    "academic_department",
    "educational_institution",
    "library",
    "preschool",
    "primary_school",
    "research_institute",
    "school",
    "secondary_school",
    "university",
]

google_education_5000 = places_nearby_grid(
    circles,
    token=token,
    R=R,
    place_types=place_types,
    field_mask=field_mask,
    sleep_sec=0.1,
)

google_education_5000

,circle_id,id,name,address,primary_type,lat,lon,business_status,geometry
0,1,ChIJ3-Y0EhUDj4cR-oSAZqjaDwA,Irene-Wakonda Jr-Sr High School,"130 State St, Irene, SD 57037, USA",secondary_school,43.085325,-97.156567,OPERATIONAL,POINT (-97.15657 43.08533)
1,3,ChIJe4yLyizljocRCjb1nlBVO-s,Centerville Public School,"610 Lincoln St, Centerville, SD 57014, USA",secondary_school,43.118777,-96.957527,OPERATIONAL,POINT (-96.95753 43.11878)
2,3,ChIJ9-nKwSzljocRxPuhWAlxF3M,Centerville Community Library,"421 Florida St, Centerville, SD 57014, USA",library,43.118708,-96.957811,OPERATIONAL,POINT (-96.95781 43.11871)
3,3,ChIJcbpupNT6jocR2Husg4wETuA,Centerville Council Room,"741 Main St, Centerville, SD 57014, USA",library,43.117006,-96.959418,OPERATIONAL,POINT (-96.95942 43.11701)
4,5,ChIJ5YePxBXzjocR4oe6E1TiFBk,Beresford High School,"301 W Maple St, Beresford, SD 57004, USA",secondary_school,43.075392,-96.777340,OPERATIONAL,POINT (-96.77734 43.07539)
...,...,...,...,...,...,...,...,...,...
288,165,ChIJl9_jRsLri4cRbcxSsGMTPKc,Southwest,"550 Elizabeth St, Edgerton, MN 56128, USA",secondary_school,43.876270,-96.136780,OPERATIONAL,POINT (-96.13678 43.87627)
289,165,ChIJv_Ueor_ri4cRoXsJphn4GSo,Edgerton Elementary School,"423 1st Ave W, Edgerton, MN 56128, USA",primary_school,43.877155,-96.130142,OPERATIONAL,POINT (-96.13014 43.87716)
290,165,ChIJs8jyNr7ri4cRXFShOD1NWJo,Edgerton Christian Elementary,"210 Elizabeth St, Edgerton, MN 56128, USA",primary_school,43.876166,-96.132045,OPERATIONAL,POINT (-96.13205 43.87617)
291,165,ChIJR0zjP5Xri4cR3YLzPB7oELg,Edgerton Public Library,"811 1st Ave W, Edgerton, MN 56128, USA",library,43.873520,-96.130402,OPERATIONAL,POINT (-96.13040 43.87352)


In [95]:
google_education_5000.to_file('/Users/houpuli/Redlining Lab Dropbox/HOUPU LI/POI research/Sioux_Falls_MSA/Sious_Falls_google_place_5000_bycat/google_education_5000.geojson', driver="GeoJSON")

In [ ]:
# google_education_5000 = gpd.read_file('/Users/houpuli/Downloads/google_place_5000_bycat/google_education_5000.geojson', driver="GeoJSON")

## 6.entertainment

In [96]:
field_mask = [
    "places.id",
    "places.displayName",
    "places.formattedAddress",
    "places.location",
    "places.primaryType",
    "places.businessStatus",
]

place_types_1 = [
    "adventure_sports_center",
    "amphitheatre",
    "amusement_center",
    "amusement_park",
    "aquarium",
    "banquet_hall",
    "barbecue_area",
    "botanical_garden",
    "bowling_alley",
    "casino",
    "childrens_camp",
    "city_park",
    "comedy_club",
    "community_center",
    "concert_hall",
    "convention_center",
    "cultural_center",
    "cycling_park",
    "dance_hall",
    "dog_park",
    "event_venue",
    "ferris_wheel",
    "garden",
    "go_karting_venue",
    "hiking_area",
    "historical_landmark",
    "indoor_playground",
    "internet_cafe",
]

place_types_2 = [
    "karaoke",
    "live_music_venue",
    "marina",
    "miniature_golf_course",
    "movie_rental",
    "movie_theater",
    "national_park",
    "night_club",
    "observation_deck",
    "off_roading_area",
    "opera_house",
    "paintball_center",
    "park",
    "philharmonic_hall",
    "picnic_ground",
    "planetarium",
    "plaza",
    "roller_coaster",
    "skateboard_park",
    "state_park",
    "tourist_attraction",
    "video_arcade",
    "vineyard",
    "visitor_center",
    "water_park",
    "wedding_venue",
    "wildlife_park",
    "wildlife_refuge",
    "zoo",
]

google_entertainment_5000_a = places_nearby_grid(
    circles, token=token, R=R,
    place_types=place_types_1,
    field_mask=field_mask, sleep_sec=0.1,
)

google_entertainment_5000_b = places_nearby_grid(
    circles, token=token, R=R,
    place_types=place_types_2,
    field_mask=field_mask, sleep_sec=0.1,
)

google_entertainment_5000 = pd.concat(
    [google_entertainment_5000_a, google_entertainment_5000_b]
).drop_duplicates(subset="id").reset_index(drop=True)

google_entertainment_5000

,circle_id,id,name,address,primary_type,lat,lon,business_status,geometry
0,1,ChIJZYKWSQADj4cR3ktaQ3tBMKA,Irene Water Tower,"401 N Clark Ave, Irene, SD 57037, USA",historical_landmark,43.083337,-97.152966,OPERATIONAL,POINT (-97.15297 43.08334)
1,4,ChIJTzUZeADzjocR71C4e4oi-ss,Casino At Truck Towne Travel,"47018 SD-46, Beresford, SD 57004, USA",casino,43.084210,-96.802832,OPERATIONAL,POINT (-96.80283 43.08421)
2,5,ChIJW55SMwDzjocRz5JcNV8xG84,Beresford Dog Park,"Beresford, SD 57004, USA",dog_park,43.077889,-96.781720,OPERATIONAL,POINT (-96.78172 43.07789)
3,8,ChIJ8dGgpBiDjocRi5VCWIoSxfI,Hudson Community Center,"120 6th St, Hudson, SD 57034, USA",community_center,43.127472,-96.457085,OPERATIONAL,POINT (-96.45708 43.12747)
4,8,ChIJMU2Ka5iDjocRVq17th0fNkQ,Mouth of the Rock River,"Hawarden, IA 51023, USA",historical_landmark,43.082302,-96.453853,OPERATIONAL,POINT (-96.45385 43.08230)
...,...,...,...,...,...,...,...,...,...
500,158,ChIJuc4dNgBHiYcRKpbrvzRHFCQ,Dell Rapids Pavillon,"601 Park St, Dell Rapids, SD 57022, USA",park,43.822096,-96.706348,OPERATIONAL,POINT (-96.70635 43.82210)
501,162,ChIJQ846hUXhi4cRq_LzBejPvtw,Jasper City Park,"113 W 3rd St, Jasper, MN 56144, USA",park,43.848846,-96.397974,OPERATIONAL,POINT (-96.39797 43.84885)
502,162,ChIJpUQibADhi4cRqTgCH41eK6M,Jasper Wildlife Management Area,"Jasper, MN 56144, USA",wildlife_refuge,43.868453,-96.369076,OPERATIONAL,POINT (-96.36908 43.86845)
503,164,ChIJDw0_NwDpi4cRG2uNblw4bSg,Poplar Creek Wildlife Management Area - South ...,"Edgerton, MN 56128, USA",wildlife_refuge,43.864657,-96.204015,OPERATIONAL,POINT (-96.20402 43.86466)


In [97]:
google_entertainment_5000.to_file('/Users/houpuli/Redlining Lab Dropbox/HOUPU LI/POI research/Sioux_Falls_MSA/Sious_Falls_google_place_5000_bycat/google_entertainment_5000.geojson', driver="GeoJSON")

In [ ]:
# google_entertainment_5000 = gpd.read_file('/Users/houpuli/Downloads/google_place_5000_bycat/google_entertainment_5000.geojson', driver="GeoJSON")

## 7.facilities

In [98]:
field_mask = [
    "places.id",
    "places.displayName",
    "places.formattedAddress",
    "places.location",
    "places.primaryType",
    "places.businessStatus",
]

place_types = [
    "public_bath",
    "public_bathroom",
    "stable",
]

google_facilities_5000 = places_nearby_grid(
    circles,
    token=token,
    R=R,
    place_types=place_types,
    field_mask=field_mask,
    sleep_sec=0.1,
)

google_facilities_5000

,circle_id,id,name,address,primary_type,lat,lon,business_status,geometry
0,19,ChIJqSmaF4ebjocRLdAuuaCHg9A,Restroom and Showers,"48187 Co Hwy 140, Canton, SD 57013, USA",public_bathroom,43.214966,-96.571920,OPERATIONAL,POINT (-96.57192 43.21497)
1,19,ChIJ14Qhkw6ZjocRdske8obkwUc,Vault toilet,"Unnamed Road, Canton, SD 57013, USA",public_bathroom,43.218914,-96.577684,OPERATIONAL,POINT (-96.57768 43.21891)
2,19,ChIJC-OZzfibjocR4SayWoz1g1I,Restroom and Showers,"28773 482nd Ave, Canton, SD 57013, USA",public_bathroom,43.217048,-96.572956,OPERATIONAL,POINT (-96.57296 43.21705)
3,19,ChIJqyAr-h2bjocRfQ0r-jleEXk,Vault toilet,"28773 482nd Ave, Canton, SD 57013, USA",public_bathroom,43.217196,-96.573843,OPERATIONAL,POINT (-96.57384 43.21720)
4,19,ChIJ7XDDceGbjocR0FaR6-yBM8A,Restroom and Showers,"28771 482nd Ave, Canton, SD 57013, USA",public_bathroom,43.217795,-96.575728,OPERATIONAL,POINT (-96.57573 43.21779)
...,...,...,...,...,...,...,...,...,...
111,130,ChIJGeuSSRXxi4cR_zEWoWHH36U,Vault toilet,"140th Ave, Luverne, MN 56156, USA",public_bathroom,43.719927,-96.192554,OPERATIONAL,POINT (-96.19255 43.71993)
112,130,ChIJnZ56Txzxi4cRhtujDri6d00,Vault toilet,"Mound Creek Trail, Luverne, MN 56156, USA",public_bathroom,43.722028,-96.191500,OPERATIONAL,POINT (-96.19150 43.72203)
113,130,ChIJHa9Oy2Txi4cR1VHR5Uwy0Go,Restroom/Showers,"PRC5+H5, Luverne, MN 56156, USA",public_bathroom,43.721496,-96.192092,OPERATIONAL,POINT (-96.19209 43.72150)
114,140,ChIJAQJAIa0wiYcRlX0KE1MJh-c,Benson Equestrian Center,"46875 252nd St, Baltic, SD 57003, USA",stable,43.729336,-96.836815,OPERATIONAL,POINT (-96.83681 43.72934)


In [99]:
google_facilities_5000.to_file('/Users/houpuli/Redlining Lab Dropbox/HOUPU LI/POI research/Sioux_Falls_MSA/Sious_Falls_google_place_5000_bycat/google_facilities_5000.geojson', driver="GeoJSON")

In [ ]:
# google_facilities_5000 = gpd.read_file('/Users/houpuli/Downloads/google_place_5000_bycat/google_facilities_5000.geojson', driver="GeoJSON")

## 8.finance

In [100]:
field_mask = [
    "places.id",
    "places.displayName",
    "places.formattedAddress",
    "places.location",
    "places.primaryType",
    "places.businessStatus",
]

place_types = [
    "accounting",
    "atm",
    "bank",
]

google_finance_5000 = places_nearby_grid(
    circles,
    token=token,
    R=R,
    place_types=place_types,
    field_mask=field_mask,
    sleep_sec=0.1,
)

google_finance_5000

,circle_id,id,name,address,primary_type,lat,lon,business_status,geometry
0,3,ChIJ459HKNX6jocRIqnnx4C7NIc,One American Bank,"549 Broadway St, Centerville, SD 57014, USA",bank,43.117345,-96.961534,OPERATIONAL,POINT (-96.96153 43.11735)
1,3,ChIJ-TRErdX6jocRcmiGgo1f7JU,ATM,"831 Broadway St, Centerville, SD 57014, USA",atm,43.114703,-96.961653,OPERATIONAL,POINT (-96.96165 43.11470)
2,4,ChIJbU_afbHzjocR6MMFU_u5nSA,ATM (Truck Towne Inc),"47016 SD-46, Beresford, SD 57004, USA",atm,43.084173,-96.802408,OPERATIONAL,POINT (-96.80241 43.08417)
3,5,ChIJ0wEBGhnzjocR0z5QnBAJMcQ,First Savings Bank,"201 N 3rd St, Beresford, SD 57004, USA",bank,43.081834,-96.774363,OPERATIONAL,POINT (-96.77436 43.08183)
4,5,ChIJFZFoexfzjocR_g2ik29c4ck,First Dakota National Bank,"1208 W Cedar St, Beresford, SD 57004, USA",bank,43.084299,-96.789432,OPERATIONAL,POINT (-96.78943 43.08430)
...,...,...,...,...,...,...,...,...,...
285,159,ChIJbWF438xAiYcRmhZ4VkVkGCQ,ATM,"500 W 4th St, Dell Rapids, SD 57022, USA",atm,43.824198,-96.724700,OPERATIONAL,POINT (-96.72470 43.82420)
286,162,ChIJO0DEuPrhi4cRymiCu3v_Bb4,Peoples Bank,"121 Wall St W, Jasper, MN 56144, USA",bank,43.849524,-96.398319,OPERATIONAL,POINT (-96.39832 43.84952)
287,162,ChIJjQIB2Prhi4cRuf-NhATxUZY,Shazam Atm,"121 Wall St W, Jasper, MN 56144, USA",atm,43.849625,-96.398257,OPERATIONAL,POINT (-96.39826 43.84962)
288,165,ChIJXQdfn5Xri4cRbTk0c3VpP6s,First State Bank Southwest,"760 Main St, Edgerton, MN 56128, USA",bank,43.873909,-96.128332,OPERATIONAL,POINT (-96.12833 43.87391)


In [101]:
google_finance_5000.to_file('/Users/houpuli/Redlining Lab Dropbox/HOUPU LI/POI research/Sioux_Falls_MSA/Sious_Falls_google_place_5000_bycat/google_finance_5000.geojson', driver="GeoJSON")

In [ ]:
# google_finance_5000 = gpd.read_file('/Users/houpuli/Downloads/google_place_5000_bycat/google_finance_5000.geojson', driver="GeoJSON")

## 9.food

In [102]:
field_mask = [
    "places.id",
    "places.displayName",
    "places.formattedAddress",
    "places.location",
    "places.primaryType",
    "places.businessStatus",
]

place_types_1 = [
    "acai_shop",
    "afghani_restaurant",
    "african_restaurant",
    "american_restaurant",
    "argentinian_restaurant",
    "asian_fusion_restaurant",
    "asian_restaurant",
    "australian_restaurant",
    "austrian_restaurant",
    "bagel_shop",
    "bakery",
    "bangladeshi_restaurant",
    "bar",
    "bar_and_grill",
    "barbecue_restaurant",
    "basque_restaurant",
    "bavarian_restaurant",
    "beer_garden",
    "belgian_restaurant",
    "bistro",
]

place_types_2 = [
    "brazilian_restaurant",
    "breakfast_restaurant",
    "brewery",
    "brewpub",
    "british_restaurant",
    "brunch_restaurant",
    "buffet_restaurant",
    "burmese_restaurant",
    "burrito_restaurant",
    "cafe",
    "cafeteria",
    "cajun_restaurant",
    "cake_shop",
    "californian_restaurant",
    "cambodian_restaurant",
    "candy_store",
    "cantonese_restaurant",
    "caribbean_restaurant",
    "cat_cafe",
    "chicken_restaurant",
]

place_types_3 = [
    "chicken_wings_restaurant",
    "chilean_restaurant",
    "chinese_noodle_restaurant",
    "chinese_restaurant",
    "chocolate_factory",
    "chocolate_shop",
    "cocktail_bar",
    "coffee_roastery",
    "coffee_shop",
    "coffee_stand",
    "colombian_restaurant",
    "confectionery",
    "croatian_restaurant",
    "cuban_restaurant",
    "czech_restaurant",
    "danish_restaurant",
    "deli",
    "dessert_restaurant",
    "dessert_shop",
    "dim_sum_restaurant",
]

place_types_4 = [
    "diner",
    "dog_cafe",
    "donut_shop",
    "dumpling_restaurant",
    "dutch_restaurant",
    "eastern_european_restaurant",
    "ethiopian_restaurant",
    "european_restaurant",
    "falafel_restaurant",
    "family_restaurant",
    "fast_food_restaurant",
    "filipino_restaurant",
    "fine_dining_restaurant",
    "fish_and_chips_restaurant",
    "fondue_restaurant",
    "food_court",
    "french_restaurant",
    "fusion_restaurant",
    "gastropub",
    "german_restaurant",
]

place_types_5 = [
    "greek_restaurant",
    "gyro_restaurant",
    "halal_restaurant",
    "hamburger_restaurant",
    "hawaiian_restaurant",
    "hookah_bar",
    "hot_dog_restaurant",
    "hot_dog_stand",
    "hot_pot_restaurant",
    "hungarian_restaurant",
    "ice_cream_shop",
    "indian_restaurant",
    "indonesian_restaurant",
    "irish_pub",
    "irish_restaurant",
    "israeli_restaurant",
    "italian_restaurant",
    "japanese_curry_restaurant",
    "japanese_izakaya_restaurant",
    "japanese_restaurant",
]

place_types_6 = [
    "juice_shop",
    "kebab_shop",
    "korean_barbecue_restaurant",
    "korean_restaurant",
    "latin_american_restaurant",
    "lebanese_restaurant",
    "lounge_bar",
    "malaysian_restaurant",
    "meal_delivery",
    "meal_takeaway",
    "mediterranean_restaurant",
    "mexican_restaurant",
    "middle_eastern_restaurant",
    "mongolian_barbecue_restaurant",
    "moroccan_restaurant",
    "noodle_shop",
    "north_indian_restaurant",
    "oyster_bar_restaurant",
    "pakistani_restaurant",
    "pastry_shop",
]

place_types_7 = [
    "persian_restaurant",
    "peruvian_restaurant",
    "pizza_delivery",
    "pizza_restaurant",
    "polish_restaurant",
    "portuguese_restaurant",
    "pub",
    "ramen_restaurant",
    "restaurant",
    "romanian_restaurant",
    "russian_restaurant",
    "salad_shop",
    "sandwich_shop",
    "scandinavian_restaurant",
    "seafood_restaurant",
    "shawarma_restaurant",
    "snack_bar",
    "soul_food_restaurant",
    "soup_restaurant",
    "south_american_restaurant",
]

place_types_8 = [
    "south_indian_restaurant",
    "southwestern_us_restaurant",
    "spanish_restaurant",
    "sports_bar",
    "sri_lankan_restaurant",
    "steak_house",
    "sushi_restaurant",
    "swiss_restaurant",
    "taco_restaurant",
    "taiwanese_restaurant",
    "tapas_restaurant",
    "tea_house",
    "tex_mex_restaurant",
    "thai_restaurant",
    "tibetan_restaurant",
    "tonkatsu_restaurant",
    "turkish_restaurant",
    "ukrainian_restaurant",
    "vegan_restaurant",
    "vegetarian_restaurant",
]

place_types_9 = [
    "vietnamese_restaurant",
    "western_restaurant",
    "wine_bar",
    "winery",
    "yakiniku_restaurant",
    "yakitori_restaurant",
]

results = []
for pt in [place_types_1, place_types_2, place_types_3, place_types_4,
           place_types_5, place_types_6, place_types_7, place_types_8, place_types_9]:
    df = places_nearby_grid(
        circles, token=token, R=R,
        place_types=pt,
        field_mask=field_mask, sleep_sec=0.1,
    )
    results.append(df)

google_food_5000 = pd.concat(results).drop_duplicates(subset="id").reset_index(drop=True)
google_food_5000

,circle_id,id,name,address,primary_type,lat,lon,business_status,geometry
0,3,ChIJe8QT1dT6jocRTOr7yYRmlrg,Zebra King Donuts,"516 Broadway St, Centerville, SD 57014, USA",donut_shop,43.117844,-96.960861,OPERATIONAL,POINT (-96.96086 43.11784)
1,3,ChIJ4yKiJ9X6jocRnYX-tnPY6Dk,Tuffy’s Bar and Grill,"601 Broadway St, Centerville, SD 57014, USA",sports_bar,43.116997,-96.961617,OPERATIONAL,POINT (-96.96162 43.11700)
2,3,ChIJyc7CMdT6jocRA8BfSrkSExo,Desert Inn,"841 State St, Centerville, SD 57014, USA",bar,43.113583,-96.960865,OPERATIONAL,POINT (-96.96086 43.11358)
3,4,ChIJb4FhdQDzjocR_dKSw8W33X0,County Line,"47106 SD-46, Beresford, SD 57004, USA",bar,43.084228,-96.803123,OPERATIONAL,POINT (-96.80312 43.08423)
4,5,ChIJKQMz-AfzjocRKW9kmGFGZJg,Bertz Sports Bar and Grill,"1406 W Cedar St, Beresford, SD 57004, USA",bar_and_grill,43.084125,-96.790796,OPERATIONAL,POINT (-96.79080 43.08412)
...,...,...,...,...,...,...,...,...,...
615,74,ChIJHzOtRT20jocRPRcnKwow01M,Saigon Panda,"3301 E 26th St, Sioux Falls, SD 57103, USA",vietnamese_restaurant,43.528648,-96.685638,OPERATIONAL,POINT (-96.68564 43.52865)
616,90,ChIJ6w0ki_20jocRG60_IcGWDSs,Lam's Vietnamese Restaurant,"1600 E Rice St, Sioux Falls, SD 57103, USA",vietnamese_restaurant,43.562395,-96.706006,OPERATIONAL,POINT (-96.70601 43.56239)
617,107,ChIJD4XwU-Y1iYcRCdhAgkrQJrs,Strawbale Winery,"47215 257th St, Renner, SD 57055, USA",winery,43.659099,-96.768083,CLOSED_TEMPORARILY,POINT (-96.76808 43.65910)
618,109,ChIJHw91w3JNiYcRqvUQey1djgg,Wilde Prairie Winery,"48052 259th St, Brandon, SD 57005, USA",winery,43.632136,-96.601002,CLOSED_TEMPORARILY,POINT (-96.60100 43.63214)


In [103]:
google_food_5000.to_file('/Users/houpuli/Redlining Lab Dropbox/HOUPU LI/POI research/Sioux_Falls_MSA/Sious_Falls_google_place_5000_bycat/google_food_5000.geojson', driver="GeoJSON")

In [ ]:
# google_food2_5000 = gpd.read_file('/Users/houpuli/Downloads/google_place_5000_bycat/google_food2_5000.geojson', driver="GeoJSON")

## 10.government

In [104]:
field_mask = [
    "places.id",
    "places.displayName",
    "places.formattedAddress",
    "places.location",
    "places.primaryType",
    "places.businessStatus",
]

place_types = [
    "city_hall",
    "courthouse",
    "embassy",
    "fire_station",
    "government_office",
    "local_government_office",
    "neighborhood_police_station",
    "police",
    "post_office",
]

google_government_5000 = places_nearby_grid(
    circles,
    token=token,
    R=R,
    place_types=place_types,
    field_mask=field_mask,
    sleep_sec=0.1,
)

google_government_5000

,circle_id,id,name,address,primary_type,lat,lon,business_status,geometry
0,0,ChIJWfM0amIbj4cRvFWqdOnl_QY,Olsen Yorkshires,"29369 451st Ave, Irene, SD 57037, USA",ranch,43.130928,-97.183382,OPERATIONAL,POINT (-97.18338 43.13093)
1,3,ChIJcbpupNT6jocREjEuDVYvpjA,Centerville City Hall,"741 Main St, Centerville, SD 57014, USA",local_government_office,43.117006,-96.959418,OPERATIONAL,POINT (-96.95942 43.11701)
2,3,ChIJiwCmLtX6jocR9yGqbevqTmM,United States Postal Service,"533 Broadway St, Centerville, SD 57014, USA",post_office,43.117570,-96.961555,OPERATIONAL,POINT (-96.96156 43.11757)
3,3,ChIJ8Zt0otz7jocRqREwHw0Ok4Q,Centerville Fire Department,"611 Iowa St, Centerville, SD 57014, USA",fire_station,43.116956,-96.959991,OPERATIONAL,POINT (-96.95999 43.11696)
4,3,ChIJcbpupNT6jocR8JirrnYpurU,Centerville Police Department,"Centerville, SD 57014, USA",government_office,43.117568,-96.959984,OPERATIONAL,POINT (-96.95998 43.11757)
...,...,...,...,...,...,...,...,...,...
436,162,ChIJN8xW3u_hi4cROJDPDGHc0ew,Natural Resources Department,"2nd St, Jasper, MN 56144, USA",government_office,43.850884,-96.395611,OPERATIONAL,POINT (-96.39561 43.85088)
437,165,ChIJ3QXSVJTri4cRWZ2gksyeutg,Edgerton Police Department,"Edgerton, MN 56128, USA",government_office,43.872468,-96.128637,OPERATIONAL,POINT (-96.12864 43.87247)
438,165,ChIJ973Jd5Xri4cRybpQ2f1BLSU,United States Postal Service,"741 Main St, Edgerton, MN 56128, USA",post_office,43.874128,-96.128919,OPERATIONAL,POINT (-96.12892 43.87413)
439,165,ChIJsW4fQJXri4cR4sSXBDszMUM,Edgerton City Clerk,"801 1st Ave W, Edgerton, MN 56128, USA",local_government_office,43.873488,-96.130400,OPERATIONAL,POINT (-96.13040 43.87349)


In [105]:
google_government_5000.to_file('/Users/houpuli/Redlining Lab Dropbox/HOUPU LI/POI research/Sioux_Falls_MSA/Sious_Falls_google_place_5000_bycat/google_government_5000.geojson', driver="GeoJSON")

In [ ]:
# google_government_5000 = gpd.read_file('/Users/houpuli/Downloads/google_place_5000_bycat/google_government_5000.geojson', driver="GeoJSON")

## 11.health

In [106]:
field_mask = [
    "places.id",
    "places.displayName",
    "places.formattedAddress",
    "places.location",
    "places.primaryType",
    "places.businessStatus",
]

place_types = [
    "chiropractor",
    "dental_clinic",
    "dentist",
    "doctor",
    "drugstore",
    "general_hospital",
    "hospital",
    "massage",
    "massage_spa",
    "medical_center",
    "medical_clinic",
    "medical_lab",
    "pharmacy",
    "physiotherapist",
    "sauna",
    "skin_care_clinic",
    "spa",
    "tanning_studio",
    "wellness_center",
    "yoga_studio",
]

google_health_5000 = places_nearby_grid(
    circles,
    token=token,
    R=R,
    place_types=place_types,
    field_mask=field_mask,
    sleep_sec=0.1,
)

google_health_5000

,circle_id,id,name,address,primary_type,lat,lon,business_status,geometry
0,3,ChIJ8w401dT6jocRlkL7Jg1Guww,Centerville Medical Clinic,"513 Broadway St, Centerville, SD 57014, USA",medical_clinic,43.117907,-96.961487,OPERATIONAL,POINT (-96.96149 43.11791)
1,3,ChIJ8w401dT6jocReExVolx6E_8,Centerville Community Pharmacy,"513 Broadway St, Centerville, SD 57014, USA",pharmacy,43.117964,-96.960999,CLOSED_TEMPORARILY,POINT (-96.96100 43.11796)
2,3,ChIJ8w401dT6jocRyNU-D0iIpPo,"Dr. Syed A. Shah, MD","512 Broadway St, Centerville, SD 57014, USA",doctor,43.117985,-96.961044,OPERATIONAL,POINT (-96.96104 43.11798)
3,3,ChIJO6qZd56hj4cRPrRzPHExFfo,Delzer Chiropractic,"540 Broadway St, Centerville, SD 57014, USA",chiropractor,43.117497,-96.960847,OPERATIONAL,POINT (-96.96085 43.11750)
4,5,ChIJtaisfhfzjocRDoMdiMthyLE,Neighborhood Dental,"711 W Cedar St, Beresford, SD 57004, USA",dentist,43.083288,-96.783574,OPERATIONAL,POINT (-96.78357 43.08329)
...,...,...,...,...,...,...,...,...,...
510,165,ChIJO_v4vKrri4cRPIRr8To9d94,"Edgerton Main Street Chiropractic, PLLC","821 Main St, Edgerton, MN 56128, USA",chiropractor,43.873156,-96.128946,OPERATIONAL,POINT (-96.12895 43.87316)
511,165,ChIJmSTuBpTri4cRva-5Otnj4Xw,Joni's Soothing Touch,"321 Mill St E, Edgerton, MN 56128, USA",massage,43.870107,-96.126241,OPERATIONAL,POINT (-96.12624 43.87011)
512,165,ChIJ5c58h3jui4cRNbrIXyV14Xs,The Cranial Quest,"2255 160th Ave, Edgerton, MN 56128, USA",medical_clinic,43.809281,-96.153156,OPERATIONAL,POINT (-96.15316 43.80928)
513,165,ChIJA5H9yqeTi4cRmpfx8mKXLBg,Graber T J DC,"837 Main St, Edgerton, MN 56128, USA",chiropractor,43.872895,-96.129056,OPERATIONAL,POINT (-96.12906 43.87289)


In [107]:
google_health_5000.to_file('/Users/houpuli/Redlining Lab Dropbox/HOUPU LI/POI research/Sioux_Falls_MSA/Sious_Falls_google_place_5000_bycat/google_health_5000.geojson', driver="GeoJSON")

In [ ]:
# google_health_5000 = gpd.read_file('/Users/houpuli/Downloads/google_place_5000_bycat/google_health_5000.geojson', driver="GeoJSON")

## 12.nature

In [108]:
field_mask = [
    "places.id",
    "places.displayName",
    "places.formattedAddress",
    "places.location",
    "places.primaryType",
    "places.businessStatus",
]

place_types = [
    "beach",
    "island",
    "lake",
    "mountain_peak",
    "nature_preserve",
    "river",
    "scenic_spot",
    "woods",
]

google_nature_5000 = places_nearby_grid(
    circles,
    token=token,
    R=R,
    place_types=place_types,
    field_mask=field_mask,
    sleep_sec=0.1,
)

google_nature_5000

,circle_id,id,name,address,primary_type,lat,lon,business_status,geometry
0,0,ChIJUSt1WhYbj4cRhpRVT1cgibM,South Dakota,"Turkey Valley Township, SD 57037, USA",lake,43.123733,-97.208371,None,POINT (-97.20837 43.12373)
1,0,ChIJTa1WRDEbj4cRcfdC-rnlcys,South Dakota,"Turkey Valley Township, SD 57037, USA",lake,43.108334,-97.206897,None,POINT (-97.20690 43.10833)
2,0,ChIJTa1WRDEbj4cRcfdC-rnlcys,South Dakota,"Turkey Valley Township, SD 57037, USA",lake,43.108334,-97.206897,None,POINT (-97.20690 43.10833)
3,0,ChIJUSt1WhYbj4cRhpRVT1cgibM,South Dakota,"Turkey Valley Township, SD 57037, USA",lake,43.123733,-97.208371,None,POINT (-97.20837 43.12373)
4,0,ChIJNas_-xYbj4cRr8vBtW-8AhQ,South Dakota,"Turkey Valley Township, SD 57037, USA",lake,43.124794,-97.208286,None,POINT (-97.20829 43.12479)
...,...,...,...,...,...,...,...,...,...
3142,165,ChIJy4SXy_nsi4cRu8xfc872hE8,Minnesota,"Battle Plain Township, MN 56128, USA",lake,43.827438,-96.107231,None,POINT (-96.10723 43.82744)
3143,165,ChIJ8bCIJnLsi4cRg9cJboon-Lo,Minnesota,"Battle Plain Township, MN 56128, USA",lake,43.847986,-96.126441,None,POINT (-96.12644 43.84799)
3144,165,ChIJU-CwPPfsi4cRFB3lRsWSNtc,Minnesota,"Battle Plain Township, MN 56128, USA",lake,43.830179,-96.106072,None,POINT (-96.10607 43.83018)
3145,165,ChIJOaRQs_nsi4cRnjAtBD0vuF8,Minnesota,"Battle Plain Township, MN 56128, USA",lake,43.826974,-96.107562,None,POINT (-96.10756 43.82697)


In [109]:
google_nature_5000.to_file('/Users/houpuli/Redlining Lab Dropbox/HOUPU LI/POI research/Sioux_Falls_MSA/Sious_Falls_google_place_5000_bycat/google_nature_5000.geojson', driver="GeoJSON")

In [ ]:
# google_nature_5000 = gpd.read_file('/Users/houpuli/Downloads/google_place_5000_bycat/google_nature_5000.geojson', driver="GeoJSON")

## 13.worship

In [110]:
field_mask = [
    "places.id",
    "places.displayName",
    "places.formattedAddress",
    "places.location",
    "places.primaryType",
    "places.businessStatus",
]

place_types = [
    "buddhist_temple",
    "church",
    "hindu_temple",
    "mosque",
    "shinto_shrine",
    "synagogue",
]


google_places_worship_5000 = places_nearby_grid(
    circles,
    token=token,
    R=R,
    place_types=place_types,
    field_mask=field_mask,
    sleep_sec=0.1,
)

google_places_worship_5000

,circle_id,id,name,address,primary_type,lat,lon,business_status,geometry
0,0,ChIJLzU7cKcaj4cR5om3bYPe4ns,Nazarene Church,"Irene, SD 57037, USA",church,43.111659,-97.264500,OPERATIONAL,POINT (-97.26450 43.11166)
1,1,ChIJ7QVS6hQDj4cR9hCVpZZ2Y50,Calvary Lutheran Church,"225 E Main St, Irene, SD 57037, USA",church,43.082530,-97.154186,OPERATIONAL,POINT (-97.15419 43.08253)
2,2,ChIJhcZmHuX8jocRy0GwHCyPM74,Immanuel Free Lutheran,"29686 458th Ave, Centerville, SD 57014, USA",church,43.085465,-97.041591,OPERATIONAL,POINT (-97.04159 43.08546)
3,3,ChIJ7cL0UirljocR6YRguQENFXo,Scandia Lutheran Church,"251 Broadway St, Centerville, SD 57014, USA",church,43.120645,-96.961638,OPERATIONAL,POINT (-96.96164 43.12064)
4,3,ChIJofGwGS3ljocRPaEqtr0Sczw,Good Shepherd Parish,"411 Wisconsin St, Centerville, SD 57014, USA",church,43.119384,-96.955561,OPERATIONAL,POINT (-96.95556 43.11938)
...,...,...,...,...,...,...,...,...,...
407,165,ChIJBQjIf5Xri4cRqmpsBtFgnsQ,First Christian Reformed Church,"120 Center St S W, Edgerton, MN 56128, USA",church,43.875027,-96.129244,OPERATIONAL,POINT (-96.12924 43.87503)
408,165,ChIJfxn5M77ri4cRIaFk_7ioklw,Bethel Christian Reform Church,"610 Main St, Edgerton, MN 56128, USA",church,43.875357,-96.128232,OPERATIONAL,POINT (-96.12823 43.87536)
409,165,ChIJc5sP7erri4cRvqYUW3K0lUE,Edgerton Protestant Reformed Church,"321 Maple St, Edgerton, MN 56128, USA",church,43.873374,-96.133726,OPERATIONAL,POINT (-96.13373 43.87337)
410,165,ChIJjcJgt-rri4cRGGOE4HYB4Eg,First Reformed: An Evangelical Presbyterian Ch...,"230 Maple St, Edgerton, MN 56128, USA",church,43.874032,-96.132130,OPERATIONAL,POINT (-96.13213 43.87403)


In [111]:
google_places_worship_5000.to_file('/Users/houpuli/Redlining Lab Dropbox/HOUPU LI/POI research/Sioux_Falls_MSA/Sious_Falls_google_place_5000_bycat/google_places_worship_5000.geojson', driver="GeoJSON")

In [ ]:
# google_places_worship_5000 = gpd.read_file('/Users/houpuli/Downloads/google_place_5000_bycat/google_places_worship_5000.geojson', driver="GeoJSON")

## 14.services

In [112]:
field_mask = [
    "places.id",
    "places.displayName",
    "places.formattedAddress",
    "places.location",
    "places.primaryType",
    "places.businessStatus",
]

place_types_1 = [
    "aircraft_rental_service",
    "association_or_organization",
    "astrologer",
    "barber_shop",
    "beautician",
    "beauty_salon",
    "body_art_service",
    "catering_service",
    "cemetery",
    "chauffeur_service",
    "child_care_agency",
    "consultant",
    "courier_service",
    "electrician",
    "employment_agency",
    "florist",
    "food_delivery",
    "foot_care",
    "funeral_home",
    "hair_care",
    "hair_salon",
]

place_types_2 = [
    "insurance_agency",
    "laundry",
    "lawyer",
    "locksmith",
    "makeup_artist",
    "marketing_consultant",
    "moving_company",
    "nail_salon",
    "non_profit_organization",
    "painter",
    "pet_boarding_service",
    "pet_care",
    "plumber",
    "psychic",
    "real_estate_agency",
    "roofing_contractor",
    "service",
    "shipping_service",
    "storage",
    "summer_camp_organizer",
    "tailor",
    "telecommunications_service_provider",
    "tour_agency",
    "tourist_information_center",
    "travel_agency",
    "veterinary_care",
]

google_services_5000_a = places_nearby_grid(
    circles, token=token, R=R,
    place_types=place_types_1,
    field_mask=field_mask, sleep_sec=0.1,
)

google_services_5000_b = places_nearby_grid(
    circles, token=token, R=R,
    place_types=place_types_2,
    field_mask=field_mask, sleep_sec=0.1,
)

google_services_5000 = pd.concat(
    [google_services_5000_a, google_services_5000_b]
).drop_duplicates(subset="id").reset_index(drop=True)

google_services_5000

,circle_id,id,name,address,primary_type,lat,lon,business_status,geometry
0,0,ChIJLzU7cKcaj4cR5om3bYPe4ns,Nazarene Church,"Irene, SD 57037, USA",church,43.111659,-97.264500,OPERATIONAL,POINT (-97.26450 43.11166)
1,1,ChIJ7QVS6hQDj4cR9hCVpZZ2Y50,Calvary Lutheran Church,"225 E Main St, Irene, SD 57037, USA",church,43.082530,-97.154186,OPERATIONAL,POINT (-97.15419 43.08253)
2,1,ChIJ-c82BjQdj4cRehN507-r_LY,Helping Hands for Haiti,"45406 295th St, Irene, SD 57037, USA",non_profit_organization,43.112792,-97.119602,OPERATIONAL,POINT (-97.11960 43.11279)
3,2,ChIJhcZmHuX8jocRy0GwHCyPM74,Immanuel Free Lutheran,"29686 458th Ave, Centerville, SD 57014, USA",church,43.085465,-97.041591,OPERATIONAL,POINT (-97.04159 43.08546)
4,3,ChIJ7cL0UirljocR6YRguQENFXo,Scandia Lutheran Church,"251 Broadway St, Centerville, SD 57014, USA",church,43.120645,-96.961638,OPERATIONAL,POINT (-96.96164 43.12064)
...,...,...,...,...,...,...,...,...,...
1881,165,ChIJsW4fQJXri4cR4sSXBDszMUM,Edgerton City Clerk,"801 1st Ave W, Edgerton, MN 56128, USA",local_government_office,43.873488,-96.130400,OPERATIONAL,POINT (-96.13040 43.87349)
1882,165,ChIJGTR7Tvvti4cRY3_NMOa_R38,Nykamp Cattle,"21 175th Ave, Edgerton, MN 56128, USA",farm,43.851419,-96.114776,OPERATIONAL,POINT (-96.11478 43.85142)
1883,165,ChIJXemMturri4cRGweug076cjw,J & J Autoworks,"215 Maple St, Edgerton, MN 56128, USA",car_repair,43.873531,-96.128132,OPERATIONAL,POINT (-96.12813 43.87353)
1884,165,ChIJ2erPjpzri4cRDe7ddYt6ea4,Hulstein Concrete & Const,"135 175th Ave, Edgerton, MN 56128, USA",general_contractor,43.868141,-96.114718,OPERATIONAL,POINT (-96.11472 43.86814)


In [113]:
google_services_5000.to_file('/Users/houpuli/Redlining Lab Dropbox/HOUPU LI/POI research/Sioux_Falls_MSA/Sious_Falls_google_place_5000_bycat/google_services_5000.geojson', driver="GeoJSON")

In [ ]:
# google_services_5000 = gpd.read_file('/Users/houpuli/Downloads/google_place_5000_bycat/google_services_5000.geojson', driver="GeoJSON")

## 15.sport

In [114]:
field_mask = [
    "places.id",
    "places.displayName",
    "places.formattedAddress",
    "places.location",
    "places.primaryType",
    "places.businessStatus",
]

place_types = [
    "arena",
    "athletic_field",
    "fishing_charter",
    "fishing_pier",
    "fishing_pond",
    "fitness_center",
    "golf_course",
    "gym",
    "ice_skating_rink",
    "indoor_golf_course",
    "playground",
    "race_course",
    "ski_resort",
    "sports_activity_location",
    "sports_club",
    "sports_coaching",
    "sports_complex",
    "sports_school",
    "stadium",
    "swimming_pool",
    "tennis_court",
]

google_sport_5000 = places_nearby_grid(
    circles,
    token=token,
    R=R,
    place_types=place_types,
    field_mask=field_mask,
    sleep_sec=0.1,
)

google_sport_5000

,circle_id,id,name,address,primary_type,lat,lon,business_status,geometry
0,0,ChIJub3T9kgDj4cRruFnI_pLDeY,Glenridge Golf Course,"45157 296th St, Irene, SD 57037, USA",golf_course,43.097211,-97.173919,OPERATIONAL,POINT (-97.17392 43.09721)
1,0,ChIJbTFfTFgDj4cRGaN-a6knyqU,IRENE FOOTBALL FIELD,"Irene, SD 57037, USA",athletic_field,43.085378,-97.163115,OPERATIONAL,POINT (-97.16312 43.08538)
2,0,ChIJS_AODLccj4cRsXhqsE-bdmE,Minnehaha Archers Club,"Co Hwy 29 & W State St, Irene, SD 57037, USA",sports_activity_location,43.085136,-97.160525,OPERATIONAL,POINT (-97.16052 43.08514)
3,3,ChIJwVRUD5rljocRWfarl4bKgC4,CENTERVILLE FOOTBALL FIELD,"Centerville, SD 57014, USA",athletic_field,43.122530,-96.954578,OPERATIONAL,POINT (-96.95458 43.12253)
4,3,ChIJEXWOWULljocR1teXoVbK0q4,Centerville Community Pool,"Centerville, SD 57014, USA",swimming_pool,43.120532,-96.954862,OPERATIONAL,POINT (-96.95486 43.12053)
...,...,...,...,...,...,...,...,...,...
454,162,ChIJX7VO84vhi4cRH7Rs5Axbf48,Tennis Courts,"Jasper, MN 56144, USA",tennis_court,43.848827,-96.392728,OPERATIONAL,POINT (-96.39273 43.84883)
455,162,ChIJy-x7--Dhi4cRM5A3VlYmve4,Beach Volleyball Court,"Jasper, MN 56144, USA",athletic_field,43.848944,-96.393635,OPERATIONAL,POINT (-96.39364 43.84894)
456,165,ChIJt0v_Vfnri4cRY4KYH48k3Og,Edgerton Football Field,"Edgerton, MN 56128, USA",athletic_field,43.874927,-96.136635,OPERATIONAL,POINT (-96.13663 43.87493)
457,165,ChIJz7WLkKPri4cREscCs2shlOU,Edgerton Swimming Pool,"1221 1st Ave W, Edgerton, MN 56128, USA",sports_activity_location,43.869409,-96.131825,OPERATIONAL,POINT (-96.13183 43.86941)


In [115]:
google_sport_5000.to_file('/Users/houpuli/Redlining Lab Dropbox/HOUPU LI/POI research/Sioux_Falls_MSA/Sious_Falls_google_place_5000_bycat/google_sport_5000.geojson', driver="GeoJSON")

In [ ]:
# google_sport_5000 = gpd.read_file('/Users/houpuli/Downloads/google_place_5000_bycat/google_sport_5000.geojson', driver="GeoJSON")

## 16.transportation

In [116]:
field_mask = [
    "places.id",
    "places.displayName",
    "places.formattedAddress",
    "places.location",
    "places.primaryType",
    "places.businessStatus",
]

place_types = [
    "airport",
    "airstrip",
    "bike_sharing_station",
    "bridge",
    "bus_station",
    "bus_stop",
    "ferry_service",
    "ferry_terminal",
    "heliport",
    "international_airport",
    "light_rail_station",
    "park_and_ride",
    "subway_station",
    "taxi_service",
    "taxi_stand",
    "toll_station",
    "train_station",
    "train_ticket_office",
    "tram_stop",
    "transit_depot",
    "transit_station",
    "transit_stop",
    "transportation_service",
    "truck_stop",
]

google_transportation_5000 = places_nearby_grid(
    circles,
    token=token,
    R=R,
    place_types=place_types,
    field_mask=field_mask,
    sleep_sec=0.1,
)

google_transportation_5000

,circle_id,id,name,address,primary_type,lat,lon,business_status,geometry
0,1,ChIJsxBE7BQDj4cRNDBplFCL2TM,Lloyd Bultsma Inc,"112 E Main St, Irene, SD 57037, USA",moving_company,43.083293,-97.157115,OPERATIONAL,POINT (-97.15712 43.08329)
1,2,ChIJP9QGsyTjjocRCF0GVgRLocs,Centerville Township Bridge Number S-18,"Centerville, SD 57014, USA",bridge,43.126834,-97.039247,OPERATIONAL,POINT (-97.03925 43.12683)
2,2,ChIJOYBXZsH8jocR6kvK6xe7UDg,Stevens Truck Services,"45846 296th St, Centerville, SD 57014, USA",moving_company,43.097634,-97.033203,OPERATIONAL,POINT (-97.03320 43.09763)
3,3,ChIJZ5SSrtD7jocRftPtw0jmtIA,CENTERVILLE SCALE,"Centerville, SD 57014, USA",transportation_service,43.113959,-96.958466,OPERATIONAL,POINT (-96.95847 43.11396)
4,4,ChIJ0Q60_a3zjocR1wRu6wROR9Q,Truck Towne Travel Plaza,"47018 SD-46, Beresford, SD 57004, USA",truck_stop,43.084147,-96.802511,OPERATIONAL,POINT (-96.80251 43.08415)
...,...,...,...,...,...,...,...,...,...
351,159,ChIJJZUCl8lAiYcR0iivfRMH28s,Lohmeyer Trucking Inc,"601 W 8th St, Dell Rapids, SD 57022, USA",moving_company,43.827749,-96.725911,OPERATIONAL,POINT (-96.72591 43.82775)
352,161,ChIJpf5NGKBZiYcRdwM52ttQ_mk,Neels Ron,"24750 487th Ave, Sherman, SD 57030, USA",moving_company,43.799240,-96.460246,OPERATIONAL,POINT (-96.46025 43.79924)
353,165,ChIJJ5EfLA7ri4cR2Fz4lBiZ1JQ,EDGERTON SCALE,"Edgerton, MN 56128, USA",transportation_service,43.871156,-96.130484,OPERATIONAL,POINT (-96.13048 43.87116)
354,165,ChIJlQivO5_ri4cRc3ZTiBieYlA,Bolluyt Trucking,"174 175th Ave, Edgerton, MN 56128, USA",moving_company,43.874000,-96.114732,OPERATIONAL,POINT (-96.11473 43.87400)


In [117]:
google_transportation_5000.to_file('/Users/houpuli/Redlining Lab Dropbox/HOUPU LI/POI research/Sioux_Falls_MSA/Sious_Falls_google_place_5000_bycat/google_transportation_5000.geojson', driver="GeoJSON")

In [ ]:
# google_transportation_5000 = gpd.read_file('/Users/houpuli/Downloads/google_place_5000_bycat/google_transportation_5000.geojson', driver="GeoJSON")

In [118]:
datasets = {
    "automotive": google_automotive_5000,
    "business": google_business_5000,
    "culture": google_culture_5000,
    "education": google_education_5000,
    "entertainment": google_entertainment_5000,
    "facilities": google_facilities_5000,
    "finance": google_finance_5000,
    "food": google_food_5000,
    "government": google_government_5000,
    "health": google_health_5000,
    "nature": google_nature_5000,
    "worship": google_places_worship_5000,
    "services": google_services_5000,
    "shop": google_shop_5000,
    "sport": google_sport_5000,
    "transportation": google_transportation_5000,
}

# Add primary_cat to each GeoDataFrame
for cat, gdf in datasets.items():
    gdf["primary_cat"] = cat

google_placescat_5000 = gpd.GeoDataFrame(
    pd.concat(list(datasets.values()), ignore_index=True),
    crs=google_automotive_5000.crs
)
google_placescat_5000

,circle_id,id,name,address,primary_type,lat,lon,business_status,geometry,primary_cat
0,3,ChIJKcdFrdX6jocRLCNhV3PTaMk,Pump N Stuff Centerville,"831 Broadway St, Centerville, SD 57014, USA",gas_station,43.114539,-96.961369,OPERATIONAL,POINT (-96.96137 43.11454),automotive
1,3,ChIJE0li1tr7jocR3jexlnq3qlk,Mr. G's Tires,"911 Garfield St, Centerville, SD 57014, USA",tire_shop,43.115963,-96.961639,OPERATIONAL,POINT (-96.96164 43.11596),automotive
2,3,ChIJyYiFINX6jocRZzmUvTNhjTM,NAPA Auto Parts - Centerville Farm & Auto Supply,"628 Broadway St, Centerville, SD 57014, USA",auto_parts_store,43.116461,-96.960883,OPERATIONAL,POINT (-96.96088 43.11646),automotive
3,3,ChIJX5Hps9X6jocROrK1BoRFQGM,Phillips 66,"831 Broadway St, Centerville, SD 57014, USA",gas_station,43.115638,-96.961478,OPERATIONAL,POINT (-96.96148 43.11564),automotive
4,3,ChIJl6N7YCnljocR4dpUp648Sx4,DK Auto,"800 North St, Centerville, SD 57014, USA",car_dealer,43.123161,-96.959936,OPERATIONAL,POINT (-96.95994 43.12316),automotive
...,...,...,...,...,...,...,...,...,...,...
11364,159,ChIJJZUCl8lAiYcR0iivfRMH28s,Lohmeyer Trucking Inc,"601 W 8th St, Dell Rapids, SD 57022, USA",moving_company,43.827749,-96.725911,OPERATIONAL,POINT (-96.72591 43.82775),transportation
11365,161,ChIJpf5NGKBZiYcRdwM52ttQ_mk,Neels Ron,"24750 487th Ave, Sherman, SD 57030, USA",moving_company,43.799240,-96.460246,OPERATIONAL,POINT (-96.46025 43.79924),transportation
11366,165,ChIJJ5EfLA7ri4cR2Fz4lBiZ1JQ,EDGERTON SCALE,"Edgerton, MN 56128, USA",transportation_service,43.871156,-96.130484,OPERATIONAL,POINT (-96.13048 43.87116),transportation
11367,165,ChIJlQivO5_ri4cRc3ZTiBieYlA,Bolluyt Trucking,"174 175th Ave, Edgerton, MN 56128, USA",moving_company,43.874000,-96.114732,OPERATIONAL,POINT (-96.11473 43.87400),transportation


In [119]:
google_placescat_5000.to_file('/Users/houpuli/Redlining Lab Dropbox/HOUPU LI/POI research/Sioux_Falls_MSA/Sious_Falls_google_place_5000_bycat/siousfalls_google_placescat_5000.geojson', driver="GeoJSON")

In [120]:
table_blist = ["administrative_area_level_3", "administrative_area_level_4", "administrative_area_level_5", "administrative_area_level_6", "administrative_area_level_7", "archipelago", "colloquial_area", "continent", "establishment", "finance", "food", "general_contractor",
"geocode", "health", "intersection", "landmark", "natural_feature", "neighborhood", "place_of_worship", "plus_code"	, "point_of_interest", "political", "postal_code_prefix", "postal_code_suffix", "postal_town", "premise", "route", "street_address", "sublocality", "sublocality_level_1",
"sublocality_level_2", "sublocality_level_3", "sublocality_level_4", "sublocality_level_5", "subpremise", "town_square"]

In [121]:
CAT_TO_TYPES = {
    "shop": [
        "asian_grocery_store", "auto_parts_store", "bicycle_store", "book_store", "building_materials_store",
        "butcher_shop", "cell_phone_store", "clothing_store", "convenience_store", "cosmetics_store",
        "department_store", "discount_store", "discount_supermarket", "electronics_store", "farmers_market",
        "flea_market", "food_store", "furniture_store", "garden_center", "general_store", "gift_shop",
        "grocery_store", "hardware_store", "health_food_store", "home_goods_store", "home_improvement_store",
        "hypermarket", "jewelry_store", "liquor_store", "market", "pet_store", "shoe_store", "shopping_mall",
        "sporting_goods_store", "sportswear_store", "store", "supermarket", "tea_store", "thrift_store",
        "toy_store", "warehouse_store", "wholesaler", "womens_clothing_store",
    ],
    "automotive": [
        "car_dealer", "car_rental", "car_repair", "car_wash", "ebike_charging_station",
        "electric_vehicle_charging_station", "gas_station", "parking", "parking_garage", "parking_lot",
        "rest_stop", "tire_shop", "truck_dealer",
    ],
    "business": [
        "business_center", "corporate_office", "coworking_space", "farm", "manufacturer", "ranch",
        "supplier", "television_studio",
    ],
    "culture": [
        "art_gallery", "art_museum", "art_studio", "auditorium", "castle", "cultural_landmark", "fountain",
        "historical_place", "history_museum", "monument", "museum", "performing_arts_theater", "sculpture",
    ],
    "education": [
        "academic_department", "educational_institution", "library", "preschool", "primary_school",
        "research_institute", "school", "secondary_school", "university",
    ],
    "entertainment": [
        "adventure_sports_center", "amphitheatre", "amusement_center", "amusement_park", "aquarium",
        "banquet_hall", "barbecue_area", "botanical_garden", "bowling_alley", "casino", "childrens_camp",
        "city_park", "comedy_club", "community_center", "concert_hall", "convention_center", "cultural_center",
        "cycling_park", "dance_hall", "dog_park", "event_venue", "ferris_wheel", "garden", "go_karting_venue",
        "hiking_area", "historical_landmark", "indoor_playground", "internet_cafe", "karaoke", "live_music_venue",
        "marina", "miniature_golf_course", "movie_rental", "movie_theater", "national_park", "night_club",
        "observation_deck", "off_roading_area", "opera_house", "paintball_center", "park", "philharmonic_hall",
        "picnic_ground", "planetarium", "plaza", "roller_coaster", "skateboard_park", "state_park",
        "tourist_attraction", "video_arcade", "vineyard", "visitor_center", "water_park", "wedding_venue",
        "wildlife_park", "wildlife_refuge", "zoo",
    ],
    "facilities": ["public_bath", "public_bathroom", "stable"],
    "finance": ["accounting", "atm", "bank"],
    "food": [
        "acai_shop", "afghani_restaurant", "african_restaurant", "american_restaurant", "argentinian_restaurant",
        "asian_fusion_restaurant", "asian_restaurant", "australian_restaurant", "austrian_restaurant", "bagel_shop",
        "bakery", "bangladeshi_restaurant", "bar", "bar_and_grill", "barbecue_restaurant", "basque_restaurant",
        "bavarian_restaurant", "beer_garden", "belgian_restaurant", "bistro", "brazilian_restaurant",
        "breakfast_restaurant", "brewery", "brewpub", "british_restaurant", "brunch_restaurant", "buffet_restaurant",
        "burmese_restaurant", "burrito_restaurant", "cafe", "cafeteria", "cajun_restaurant", "cake_shop",
        "californian_restaurant", "cambodian_restaurant", "candy_store", "cantonese_restaurant", "caribbean_restaurant",
        "cat_cafe", "chicken_restaurant", "chicken_wings_restaurant", "chilean_restaurant", "chinese_noodle_restaurant",
        "chinese_restaurant", "chocolate_factory", "chocolate_shop", "cocktail_bar", "coffee_roastery", "coffee_shop",
        "coffee_stand", "colombian_restaurant", "confectionery", "croatian_restaurant", "cuban_restaurant",
        "czech_restaurant", "danish_restaurant", "deli", "dessert_restaurant", "dessert_shop", "dim_sum_restaurant",
        "diner", "dog_cafe", "donut_shop", "dumpling_restaurant", "dutch_restaurant", "eastern_european_restaurant",
        "ethiopian_restaurant", "european_restaurant", "falafel_restaurant", "family_restaurant", "fast_food_restaurant",
        "filipino_restaurant", "fine_dining_restaurant", "fish_and_chips_restaurant", "fondue_restaurant", "food_court",
        "french_restaurant", "fusion_restaurant", "gastropub", "german_restaurant", "greek_restaurant",
        "gyro_restaurant", "halal_restaurant", "hamburger_restaurant", "hawaiian_restaurant", "hookah_bar",
        "hot_dog_restaurant", "hot_dog_stand", "hot_pot_restaurant", "hungarian_restaurant", "ice_cream_shop",
        "indian_restaurant", "indonesian_restaurant", "irish_pub", "irish_restaurant", "israeli_restaurant",
        "italian_restaurant", "japanese_curry_restaurant", "japanese_izakaya_restaurant", "japanese_restaurant",
        "juice_shop", "kebab_shop", "korean_barbecue_restaurant", "korean_restaurant", "latin_american_restaurant",
        "lebanese_restaurant", "lounge_bar", "malaysian_restaurant", "meal_delivery", "meal_takeaway",
        "mediterranean_restaurant", "mexican_restaurant", "middle_eastern_restaurant", "mongolian_barbecue_restaurant",
        "moroccan_restaurant", "noodle_shop", "north_indian_restaurant", "oyster_bar_restaurant", "pakistani_restaurant",
        "pastry_shop", "persian_restaurant", "peruvian_restaurant", "pizza_delivery", "pizza_restaurant",
        "polish_restaurant", "portuguese_restaurant", "pub", "ramen_restaurant", "restaurant", "romanian_restaurant",
        "russian_restaurant", "salad_shop", "sandwich_shop", "scandinavian_restaurant", "seafood_restaurant",
        "shawarma_restaurant", "snack_bar", "soul_food_restaurant", "soup_restaurant", "south_american_restaurant",
        "south_indian_restaurant", "southwestern_us_restaurant", "spanish_restaurant", "sports_bar",
        "sri_lankan_restaurant", "steak_house", "sushi_restaurant", "swiss_restaurant", "taco_restaurant",
        "taiwanese_restaurant", "tapas_restaurant", "tea_house", "tex_mex_restaurant", "thai_restaurant",
        "tibetan_restaurant", "tonkatsu_restaurant", "turkish_restaurant", "ukrainian_restaurant", "vegan_restaurant",
        "vegetarian_restaurant", "vietnamese_restaurant", "western_restaurant", "wine_bar", "winery",
        "yakiniku_restaurant", "yakitori_restaurant",
    ],
    "government": [
        "city_hall", "courthouse", "embassy", "fire_station", "government_office",
        "local_government_office", "neighborhood_police_station", "police", "post_office",
    ],
    "health": [
        "chiropractor", "dental_clinic", "dentist", "doctor", "drugstore", "general_hospital", "hospital",
        "massage", "massage_spa", "medical_center", "medical_clinic", "medical_lab", "pharmacy",
        "physiotherapist", "sauna", "skin_care_clinic", "spa", "tanning_studio", "wellness_center", "yoga_studio",
    ],
    "nature": ["beach", "island", "lake", "mountain_peak", "nature_preserve", "river", "scenic_spot", "woods"],
    "worship": ["buddhist_temple", "church", "hindu_temple", "mosque", "shinto_shrine", "synagogue"],
    "services": [
        "aircraft_rental_service", "association_or_organization", "astrologer", "barber_shop", "beautician",
        "beauty_salon", "body_art_service", "catering_service", "cemetery", "chauffeur_service", "child_care_agency",
        "consultant", "courier_service", "electrician", "employment_agency", "florist", "food_delivery", "foot_care",
        "funeral_home", "hair_care", "hair_salon", "insurance_agency", "laundry", "lawyer", "locksmith",
        "makeup_artist", "marketing_consultant", "moving_company", "nail_salon", "non_profit_organization", "painter",
        "pet_boarding_service", "pet_care", "plumber", "psychic", "real_estate_agency", "roofing_contractor",
        "service", "shipping_service", "storage", "summer_camp_organizer", "tailor",
        "telecommunications_service_provider", "tour_agency", "tourist_information_center", "travel_agency",
        "veterinary_care",
    ],
    "sport": [
        "arena", "athletic_field", "fishing_charter", "fishing_pier", "fishing_pond", "fitness_center",
        "golf_course", "gym", "ice_skating_rink", "indoor_golf_course", "playground", "race_course", "ski_resort",
        "sports_activity_location", "sports_club", "sports_coaching", "sports_complex", "sports_school",
        "stadium", "swimming_pool", "tennis_court",
    ],
    "transportation": [
        "airport", "airstrip", "bike_sharing_station", "bridge", "bus_station", "bus_stop", "ferry_service",
        "ferry_terminal", "heliport", "international_airport", "light_rail_station", "park_and_ride",
        "subway_station", "taxi_service", "taxi_stand", "toll_station", "train_station", "train_ticket_office",
        "tram_stop", "transit_depot", "transit_station", "transit_stop", "transportation_service", "truck_stop",
    ],
}

df = google_placescat_5000.copy()
dup_mask = df.duplicated(subset="id", keep=False)

def is_valid_type(row, cat_to_types):
    cat = row["primary_cat"]
    ptype = row["primary_type"]
    if cat not in cat_to_types:
        return False
    return ptype in cat_to_types[cat]

df["valid_type"] = True
df.loc[dup_mask, "valid_type"] = df.loc[dup_mask].apply(is_valid_type, axis=1, cat_to_types=CAT_TO_TYPES)
df_clean = df[(~dup_mask) | (df["valid_type"])].drop(columns="valid_type")

df_clean = df_clean.drop_duplicates(subset="id", keep="first")
df_clean = df_clean[~df_clean["primary_type"].isin(table_blist)].reset_index(drop=True)

# merge with google_naics_mapping to get naics_code and naics_definition
google_naics_mapping = pd.read_csv('/Users/houpuli/Redlining Lab Dropbox/HOUPU LI/POI research/mapping_google_naics.csv')
df_clean = df_clean.dropna(subset=['primary_type'])
df_clean = df_clean.merge(google_naics_mapping[['SubCategory','naics_code','naics_definition']], left_on = 'primary_type', right_on='SubCategory', how="left")
df_clean['addr_simple'] = df_clean['address'].str.split(',', n=1).str[0]
df_clean = df_clean.drop(columns=['SubCategory']).reset_index(drop=True)
df_clean

,circle_id,id,name,address,primary_type,lat,lon,business_status,geometry,primary_cat,naics_code,naics_definition,addr_simple
0,3,ChIJKcdFrdX6jocRLCNhV3PTaMk,Pump N Stuff Centerville,"831 Broadway St, Centerville, SD 57014, USA",gas_station,43.114539,-96.961369,OPERATIONAL,POINT (-96.96137 43.11454),automotive,447,Gasoline Stations,831 Broadway St
1,3,ChIJE0li1tr7jocR3jexlnq3qlk,Mr. G's Tires,"911 Garfield St, Centerville, SD 57014, USA",tire_shop,43.115963,-96.961639,OPERATIONAL,POINT (-96.96164 43.11596),automotive,4413,Automotive Parts Accessories and Tire Stores,911 Garfield St
2,3,ChIJX5Hps9X6jocROrK1BoRFQGM,Phillips 66,"831 Broadway St, Centerville, SD 57014, USA",gas_station,43.115638,-96.961478,OPERATIONAL,POINT (-96.96148 43.11564),automotive,447,Gasoline Stations,831 Broadway St
3,3,ChIJl6N7YCnljocR4dpUp648Sx4,DK Auto,"800 North St, Centerville, SD 57014, USA",car_dealer,43.123161,-96.959936,OPERATIONAL,POINT (-96.95994 43.12316),automotive,441,Motor Vehicle and Parts Dealers,800 North St
4,3,ChIJGxQWbyH7jocRWlFJUF-xsJA,S & M Auto Sales,"741 Broadway St, Centerville, SD 57014, USA",car_dealer,43.115505,-96.961723,OPERATIONAL,POINT (-96.96172 43.11550),automotive,441,Motor Vehicle and Parts Dealers,741 Broadway St
...,...,...,...,...,...,...,...,...,...,...,...,...,...
7007,158,ChIJ3-04FQBBiYcRnVDCGOd9U_I,Helipad Hospital,"Dell Rapids, SD 57022, USA",heliport,43.828568,-96.717325,OPERATIONAL,POINT (-96.71733 43.82857),transportation,4881,Support Activities for Air Transportation,Dell Rapids
7008,158,ChIJBVsqSSxHiYcRgbTkluj587U,"Heidenreich Transport, LLC","24666 475th Ave, Dell Rapids, SD 57022, USA",transportation_service,43.809930,-96.709008,OPERATIONAL,POINT (-96.70901 43.80993),transportation,4859,Other Transit and Ground Passenger Transportation,24666 475th Ave
7009,165,ChIJJ5EfLA7ri4cR2Fz4lBiZ1JQ,EDGERTON SCALE,"Edgerton, MN 56128, USA",transportation_service,43.871156,-96.130484,OPERATIONAL,POINT (-96.13048 43.87116),transportation,4859,Other Transit and Ground Passenger Transportation,Edgerton
7010,165,ChIJlQivO5_ri4cRc3ZTiBieYlA,Bolluyt Trucking,"174 175th Ave, Edgerton, MN 56128, USA",moving_company,43.874000,-96.114732,OPERATIONAL,POINT (-96.11473 43.87400),transportation,4841,General Freight Trucking,174 175th Ave


In [122]:
df_clean.to_file('/Users/houpuli/Redlining Lab Dropbox/HOUPU LI/POI research/Sioux_Falls_MSA/Sious_Falls_msa_poi/siousfalls_gplc.geojson', driver="GeoJSON")

# Google VS Overture

In [132]:
import os
import json
import base64
import numpy as np
import pandas as pd
import geopandas as gpd
from typing import Optional
from pyiceberg.expressions import And, GreaterThanOrEqual, LessThanOrEqual
from pyiceberg.catalog import load_catalog

In [133]:
import numpy as np
import geopandas as gpd
from scipy.spatial import cKDTree

def search_spatial_candidates(
    reference_gdf: gpd.GeoDataFrame,
    compared_gdf: gpd.GeoDataFrame,
    k: int = 100,
    max_dist: float = 1000, 
    id_col: str = "id",
    crs_for_distance: int = 3857,
):
    """
    Attach k nearest compared POI ids & distances to reference_gdf.

    Returns
    -------
    GeoDataFrame with two new columns:
    - cand_ids   : list of compared ids
    - cand_dist_m: list of distances (meters)
    """

    ref_proj = reference_gdf.to_crs(crs_for_distance)
    cmp_proj = compared_gdf.to_crs(crs_for_distance)

    ref_xy = np.column_stack([ref_proj.geometry.x, ref_proj.geometry.y])
    cmp_xy = np.column_stack([cmp_proj.geometry.x, cmp_proj.geometry.y])

    tree = cKDTree(cmp_xy)
    k_eff = min(k, len(compared_gdf))

    dist, idx = tree.query(ref_xy, k=k_eff)

    if k_eff == 1:
        dist = dist.reshape(-1, 1)
        idx = idx.reshape(-1, 1)

    cmp_ids = compared_gdf[id_col].to_numpy()

    cand_ids = []
    cand_dists = []

    for row_idx, row_dist in zip(idx, dist):
        ids = []
        dists = []

        for j, d in zip(row_idx, row_dist):
            if d <= max_dist:
                ids.append(cmp_ids[j])
                dists.append(d)

        cand_ids.append(ids)
        cand_dists.append(dists)

    result = reference_gdf.copy()
    result["cand_ids"] = cand_ids
    result["cand_dist_m"] = cand_dists

    return result

In [134]:
FOOD_WORDS = {
    "RESTAURANT","RESTURANT","RESTARAUNT",
    "CAFE","CAFÉ","COFFEE","BAR","BISTRO","GRILL",
    "KITCHEN","DINER","EATERY","STEAKHOUSE",
    "SEAFOOD","BUFFET","BBQ","PIZZA","SUSHI","RAMEN",
    "NOODLE","NOODLES","BURGER","BURGERS","TACO","TACOS",
    "CHICKEN","WINGS","BAKERY","DELI","DELICATESSEN",
    "COURT","FOOD","EXPRESS","HOUSE","SHOP"
}

PLACE_WORDS = {
    "HALL","CENTER","CENTRE","PLAZA","MARKET","MALL",
    "GARDEN","GARDENS","PARK","SQUARE","TOWER","TOWERS",
    "STATION","TERMINAL","BUILDING","GALLERY","THEATER","SCHOOL","COURT","INN",
    "HOTEL","MOTEL","INN","SUITES","SUITE",
    "SPA","SALON","STUDIO","CENTER","CENTRE",
    "CLUB","LOUNGE","STATION","STOP"
}

LEGAL_WORDS = {
    "LLC","INC","CORP","CORPORATION","CO","COMPANY",
    "LTD","LIMITED","GROUP","HOLDINGS","OFFICE"
}

GRAMMAR = {
    "THE","OF","AT","AND","FOR","IN","ON","BY","WITH","TO","FROM"
}

NON_PRIMARY_TOKENS = FOOD_WORDS | PLACE_WORDS | LEGAL_WORDS | GRAMMAR

In [135]:
from rapidfuzz import process, fuzz
import pandas as pd
import re
import unicodedata


def clean_name(s):
    if not isinstance(s, str):
        return ""

    s = unicodedata.normalize("NFKD", s)
    s = "".join(c for c in s if not unicodedata.combining(c)) # 1. unicode normalize (remove accents)
    s = s.upper() # 2. uppercase
    s = re.sub(r"\([^)]*\)", "", s) 
    s = re.sub(r"\b'S\b", "", s) # new change
    s = re.sub(r"\bS\b", "", s) # new change
    s = s.encode("ascii", errors="ignore").decode() # 4. remove emoji / non ascii
    s = re.sub(r"[^\w\s]", " ", s) # 5. replace special chars with space
    s = re.sub(r"\s+", " ", s) # 6. collapse spaces

    return s.strip()

def extract_prinmary_str(name):

    tokens = name.split()
    core = [t for t in tokens if t not in NON_PRIMARY_TOKENS]
    if len(core) == 1 and len(core[0]) < 3:
        return name
    if core:
        return " ".join(core)
    return name

def match_by_name(
    reference_gdf: gpd.GeoDataFrame,
    compared_gdf: gpd.GeoDataFrame,
    re_name_col: str = "name",
    comp_name_col: str = "name",
    comp_id: str = "id",
    comp_id_col: str="cat_main",
    threshold: int = 80,
):
    """
    Perform WRatio name matching within spatial candidates.

    Returns
    -------
    GeoDataFrame with:
    - matched_id_name
    - name_score
    """

    # clean names for matching
    id_to_name_clean = compared_gdf.set_index(comp_id)[comp_name_col].apply(clean_name).apply(extract_prinmary_str).to_dict()
    # raw names for storage
    id_to_name_raw = compared_gdf.set_index(comp_id)[comp_name_col].to_dict()
    # raw compared df category
    id_to_cat = compared_gdf.set_index(comp_id)[comp_id_col].to_dict()

    matched_ids = []
    scores = []
    loc_dists = []
    matched_names = []
    matched_cats = []

    for _, row in reference_gdf.iterrows():
        query = extract_prinmary_str(clean_name(row.get(re_name_col)))

        if not isinstance(query, str) or not row["cand_ids"]:
            matched_ids.append(pd.NA)
            scores.append(pd.NA)
            loc_dists.append(pd.NA)
            matched_names.append(pd.NA)
            matched_cats.append(pd.NA)
            continue

        cand_names = [id_to_name_clean.get(cid, "") for cid in row["cand_ids"]]

        top_matches = process.extract(
            query,
            cand_names,
            scorer=fuzz.WRatio,
            limit=5
        )

        best_score = -1
        best_pos = None

        for name, wr, pos in top_matches:

            pr = fuzz.partial_ratio(query, name)
            ts = fuzz.token_sort_ratio(query, name)
            tset = fuzz.token_set_ratio(query, name)

            combined = max(wr, pr, ts, tset)

            if combined > best_score:
                best_score = combined
                best_pos = pos

        score = best_score

        if score >= threshold:
            matched_ids.append(row["cand_ids"][best_pos])
            scores.append(score)
            loc_dists.append(row["cand_dist_m"][best_pos])
            matched_names.append(id_to_name_raw.get(row["cand_ids"][best_pos]))
            matched_cats.append(id_to_cat.get(row["cand_ids"][best_pos]))
        else:
            matched_ids.append(pd.NA)
            scores.append(score)
            loc_dists.append(pd.NA)
            matched_names.append(pd.NA)
            matched_cats.append(pd.NA)


    result = reference_gdf.copy()
    result["matched_id"] = matched_ids
    result["name_score"] = scores
    result["location_distance"] = loc_dists
    result["matched_name"] = matched_names
    result["matched_cat_main"] = matched_cats

    return result

In [136]:
import pandas as pd
import geopandas as gpd
from rapidfuzz import fuzz
import re
import unicodedata

def clean_name(s):
    if not isinstance(s, str):
        return ""

    s = unicodedata.normalize("NFKD", s)
    s = "".join(c for c in s if not unicodedata.combining(c)) # 1. unicode normalize (remove accents)
    s = s.upper() # 2. uppercase
    s = re.sub(r"\([^)]*\)", "", s) 
    s = s.encode("ascii", errors="ignore").decode() # 4. remove emoji / non ascii
    s = re.sub(r"[^\w\s]", " ", s) # 5. replace special chars with space
    s = re.sub(r"\s+", " ", s) # 6. collapse spaces

    return s.strip()

def address_score_check(
    reference_gdf: gpd.GeoDataFrame,
    compared_gdf: gpd.GeoDataFrame,
    addr_col_ref: str = "addr_simple",
    addr_col_cmp: str = "address",
    matched_id_col: str = "matched_id",
    id_col: str = "id",
    out_col: str = "address_score",
    out_addr_col: str = "matched_address", 
):
    """
    Compute address similarity score (0-100) for already-matched rows.

    Only rows with non-null `matched_id_col` will be scored.
    Others will have NaN.

    Returns
    -------
    GeoDataFrame with a new column `out_col`.
    """

    # map: compared id -> compared address
    id_to_addr_clean = compared_gdf.set_index(id_col)[addr_col_cmp].apply(clean_name).to_dict()
    id_to_addr_raw = compared_gdf.set_index(id_col)[addr_col_cmp].to_dict()

    scores = []
    matched_addrs = []

    for _, row in reference_gdf.iterrows():
        matched_id = row.get(matched_id_col)

        # no matched id -> no score
        if pd.isna(matched_id):
            scores.append(pd.NA)
            matched_addrs.append(pd.NA)
            continue

        addr_ref = clean_name(row.get(addr_col_ref))
        addr_cmp = id_to_addr_clean.get(matched_id)

        if isinstance(addr_cmp, str) and addr_cmp.strip():
            matched_addrs.append(id_to_addr_raw.get(matched_id))
        else:
            matched_addrs.append(pd.NA)

        # missing address on either side -> no score
        if not isinstance(addr_ref, str) or not addr_ref.strip():
            scores.append(pd.NA)
            continue
        if not isinstance(addr_cmp, str) or not addr_cmp.strip():
            scores.append(pd.NA)
            continue

        wr = fuzz.WRatio(addr_ref, addr_cmp)
        pr = fuzz.partial_ratio(addr_ref, addr_cmp)
        ts = fuzz.token_sort_ratio(addr_ref, addr_cmp)
        tset = fuzz.token_set_ratio(addr_ref, addr_cmp)

        scores.append(int(max(wr, pr, ts, tset)))

    result = reference_gdf.copy()
    result[out_col] = scores
    result[out_addr_col] = matched_addrs
    return result

In [137]:
import pandas as pd
import numpy as np
import torch
from sentence_transformers import SentenceTransformer
from tqdm import tqdm

def calculate_similarity_check(
    df, 
    cat_col_ref: str = "primary_type", 
    cat_col_cmp: str = "matched_cat_main", 
    id_col: str = "matched_id", 
    result_col: str =  "category_sim",
):

    # 1. Setup Device
    device = "mps" if torch.backends.mps.is_available() else "cpu"
    model = SentenceTransformer('all-MiniLM-L6-v2', device=device)
    
    # 2. Primary Gatekeeper: matched_id must be present
    mask = df[id_col].notna() & (df[id_col].astype(str).str.strip() != "")
    
    # 3. Create a helper to handle potential Nulls in text columns
    temp_df = df[mask].copy()
    
    # Identify where text is actually missing within the masked rows
    text_missing_mask = temp_df[cat_col_ref].isna() | temp_df[cat_col_cmp].isna()
    
    # Fill NaNs with empty strings just for the encoding step
    texts_1 = temp_df[cat_col_ref].fillna("").astype(str).tolist()
    texts_2 = temp_df[cat_col_cmp].fillna("").astype(str).tolist()

    print(f"Encoding {len(temp_df)} rows...")

    # 4. Batch Encoding
    emb1 = model.encode(texts_1, batch_size=256, show_progress_bar=True, convert_to_tensor=True)
    emb2 = model.encode(texts_2, batch_size=256, show_progress_bar=True, convert_to_tensor=True)

    # 5. Calculate Similarity
    scores = torch.nn.functional.cosine_similarity(emb1, emb2, dim=1).cpu().numpy()
    
    # 6. Apply NaN to rows where text was missing
    # Even if we encoded an empty string, the result isn't "real" if data was missing
    scores[text_missing_mask.values] = np.nan

    # 7. Map back to original dataframe
    df[result_col] = np.nan
    df.loc[mask, result_col] = scores
    
    return df

In [213]:
gplc = gpd.read_file('/Users/houpuli/Redlining Lab Dropbox/HOUPU LI/POI research/Sioux_Falls_MSA/Sious_Falls_msa_poi/siousfalls_gplc.geojson')

In [214]:
ove = gpd.read_file('/Users/houpuli/Redlining Lab Dropbox/HOUPU LI/POI research/Sioux_Falls_MSA/Sious_Falls_msa_poi/siousfalls_ove.geojson', driver='GeoJSON')

In [215]:
gplc_ove = search_spatial_candidates(reference_gdf=gplc, compared_gdf=ove, k=100)

In [216]:
gplc_ove = match_by_name(reference_gdf=gplc_ove, compared_gdf=ove, re_name_col = "name", comp_name_col = "name", threshold=80)

In [217]:
gplc_ove = address_score_check(reference_gdf=gplc_ove, compared_gdf=ove)

In [ ]:
# gplc_ove = calculate_similarity_check(gplc_ove, cat_col_ref = "naics_definition", cat_col_cmp = "matched_cat_main", id_col = "matched_id", result_col =  "category_sim")

Encoding 41762 rows...


Batches:   0%|          | 0/164 [00:00<?, ?it/s]

Batches:   0%|          | 0/164 [00:00<?, ?it/s]

In [218]:
# transfer the X and Y into float type and deal with the address score
cols_to_fix = ['name_score', 'location_distance', 'address_score']
for col in cols_to_fix:
    gplc_ove[col] = pd.to_numeric(gplc_ove[col], errors='coerce')
gplc_ove['address_score'] = gplc_ove['address_score'].fillna(0)
# gplc_ove['category_sim'] = gplc_ove['category_sim'].fillna(0)

#### Output the matched samples

In [ ]:
df = gplc_ove[gplc_ove["matched_id"].notna()].copy()

N = 1000

weights = df["primary_cat"].map(
    df["primary_cat"].value_counts(normalize=True)
)

df_sample = df.sample(
    n=N,
    weights=weights,
    random_state=42
)
df_sample[['id','name','addr_simple','primary_type','matched_id','matched_name','matched_address','matched_cat_main','location_distance']].to_csv('/Users/houpuli/Redlining Lab Dropbox/HOUPU LI/POI research/Sioux_Falls_MSA/Sious_Falls_google_comparison/siousfalls_gplc_ove_1000_sample.csv', index=False)

#### Input the matched samples

In [220]:
df_sample_check = pd.read_csv('/Users/houpuli/Redlining Lab Dropbox/HOUPU LI/POI research/Sioux_Falls_MSA/Sious_Falls_google_comparison/siousfalls_gplc_ove_1000_sample.csv')
df_sample_check = df_sample_check.drop(columns='location_distance')
df_sample_check['is_true'] = df_sample_check['is_true'].fillna(df_sample_check['is_true'])
df_sample_check = df_sample_check.merge(gplc_ove[['id','name_score','location_distance','address_score']], left_on="id", right_on="id", how="left")

In [221]:
df_sample_check['is_true'].value_counts()

is_true
1    884
0    116
Name: count, dtype: int64

In [222]:
df = df_sample_check.copy()

In [223]:
from sklearn.preprocessing import StandardScaler

X = df[['name_score', 'location_distance', 'address_score']]
y = df['is_true']

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

In [276]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y,
    test_size=0.25,
    random_state=42,
    stratify=y  # keep the same proportion of True vs False in training set and test set
)

In [277]:
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import classification_report, roc_auc_score

gb_clf = GradientBoostingClassifier()
gb_clf.fit(X_train, y_train)

gb_y_pred = gb_clf.predict(X_test)
gb_y_prob = gb_clf.predict_proba(X_test)[:, 1]

gb_cr = classification_report(y_test, gb_y_pred)
gb_auc = roc_auc_score(y_test, gb_y_prob)

print(gb_cr)
print("GradientBoosting of AUC:", gb_auc)

              precision    recall  f1-score   support

         0.0       0.87      0.80      0.83        60
         1.0       0.97      0.98      0.98       440

    accuracy                           0.96       500
   macro avg       0.92      0.89      0.91       500
weighted avg       0.96      0.96      0.96       500

GradientBoosting of AUC: 0.9865530303030303


In [280]:
mask = ny_gplc_ove['matched_id'].notnull()
df_pred = ny_gplc_ove.loc[mask].copy()

feature_cols = ['name_score', 'location_distance', 'address_score', 'category_sim']

X_new = scaler.transform(df_pred[feature_cols])
df_pred['true_match_prob'] = gb_clf.predict_proba(X_new)[:, 1]
df_pred['is_true_match'] = df_pred['true_match_prob'] >= 0.5

ny_gplc_ove.loc[mask, 'is_true_match'] = df_pred['is_true_match']
ny_gplc_ove.loc[mask, 'true_match_prob'] = df_pred['true_match_prob']

In [284]:
ny_gplc_ove['is_true_match'].value_counts()

is_true_match
True     46493
False     7145
Name: count, dtype: int64

In [340]:
ny_gplc_ove.info()

<class 'geopandas.geodataframe.GeoDataFrame'>
RangeIndex: 65680 entries, 0 to 65679
Data columns (total 24 columns):
 #   Column             Non-Null Count  Dtype   
---  ------             --------------  -----   
 0   circle_id          65680 non-null  int64   
 1   id                 65680 non-null  object  
 2   name               65680 non-null  object  
 3   address            65680 non-null  object  
 4   primary_type       65680 non-null  object  
 5   lat                65680 non-null  float64 
 6   lon                65680 non-null  float64 
 7   primary_cat        65680 non-null  object  
 8   geometry           65680 non-null  geometry
 9   naics_code         65680 non-null  float64 
 10  naics_definition   65680 non-null  object  
 11  addr_simple        65680 non-null  object  
 12  cand_ids           65680 non-null  object  
 13  cand_dist_m        65680 non-null  object  
 14  matched_id         53638 non-null  object  
 15  name_score         65290 non-null  float64 
 

# Google VS Safegraph

In [ ]:
# import os
# path = '/Users/houpuli/Redlining Lab Dropbox/HOUPU LI/POI research/Dallas_MSA/dallas_msa_poi/dallas_sf.parquet'
# size_gb = os.path.getsize(path) / 1e9
# print(f"文件大小: {size_gb:.2f} GB")

文件大小: 0.04 GB


In [193]:
sf = gpd.read_parquet('/Users/houpuli/Redlining Lab Dropbox/HOUPU LI/POI research/Sioux_Falls_MSA/Sious_Falls_msa_poi/siousfalls_sf.parquet')
sf

,PLACEKEY,PARENT_PLACEKEY,LOCATION_NAME,TRACKING_CLOSED_SINCE,TOP_CATEGORY,SUB_CATEGORY,CATEGORY_TAGS,NAICS_CODE,STREET_ADDRESS,geometry
0,224-222@5nw-9bp-vmk,None,Track-Side Service,2019-07-01,Automotive Repair and Maintenance,General Automotive Repair,"[""Auto Repair""]",811111,110 S Blair St,POINT (-96.60001 43.30049)
1,224-22n@5nw-9n4-3nq,None,Your Unique Salon,2019-07-01,Personal Care Services,"Hair, Nail, and Skin Care Services","[""Spa""]",81211,4610 S Technopolis Dr,POINT (-96.77439 43.50234)
2,227-223@5nw-9ts-w49,None,Royal Flush On Cliff,2019-07-01,Gambling Industries,Casinos (except Casino Hotels),[],713210,803 S Cliff Ave,POINT (-96.71159 43.53915)
3,227-224@5nw-9pw-fpv,None,Sioux Falls Hamer Noh Kidanemhiret Ethiopian O...,2019-07-01,Religious Organizations,Religious Organizations,"[""Church""]",813110,1609 W 11th St,POINT (-96.74904 43.54441)
4,222-222@5nw-9pv-rhq,None,Roll'n Pin Caf & Grille + Catering,2019-07-01,Restaurants and Other Eating Places,Full-Service Restaurants,"[""American Food"",""Casual Dining"",""Sandwiches""]",722511,3015 W Russell St,POINT (-96.77035 43.56804)
...,...,...,...,...,...,...,...,...,...,...
21059,225-222@5nw-8dt-t5f,None,State of South Dakota Army National Guard,2019-07-01,"Executive, Legislative, and Other General Gove...",None,[],9211,740 N Peck St,POINT (-97.38103 43.73092)
21060,22d-225@5nw-9n9-vmk,235-222@5nw-9n9-vmk,Yiting Zhou,2019-07-01,Offices of Other Health Practitioners,Offices of All Other Miscellaneous Health Prac...,[],621399,3220 S Western Ave Ste 1,POINT (-96.75074 43.51597)
21061,22f-222@5nw-9tw-v75,None,Apartments at 1515 S Norton Ave,2019-07-01,Lessors of Real Estate,Lessors of Residential Buildings and Dwellings,[],531110,1515 S Norton Ave,POINT (-96.73529 43.53143)
21062,223-238@5nw-9tw-w49,224-22k@5nw-9pw-nt9,Elizabeth Kobriger Ln,2019-07-01,Offices of Other Health Practitioners,Offices of All Other Miscellaneous Health Prac...,[],621399,1305 W 18th St,POINT (-96.74252 43.53517)


In [194]:
gplc_sf = search_spatial_candidates(reference_gdf=gplc, compared_gdf=sf, id_col = "PLACEKEY", k=100)

In [195]:
gplc_sf = match_by_name(reference_gdf=gplc_sf, compared_gdf=sf, re_name_col = "name", comp_name_col = "LOCATION_NAME", comp_id = "PLACEKEY", comp_id_col ="TOP_CATEGORY",  threshold=80)

In [196]:
gplc_sf = address_score_check(reference_gdf=gplc_sf, compared_gdf=sf, addr_col_ref = "addr_simple", addr_col_cmp = "STREET_ADDRESS", id_col = "PLACEKEY",)

In [ ]:
# dallas_gplc_sf = calculate_similarity_check(dallas_gplc_sf, cat_col_ref = "naics_definition", cat_col_cmp = "matched_cat_main", id_col = "matched_id", result_col =  "category_sim")

Encoding 42123 rows...


Batches:   0%|          | 0/165 [00:00<?, ?it/s]

Batches:   0%|          | 0/165 [00:00<?, ?it/s]

In [197]:
# transfer the X and Y into float type and deal with the address score
cols_to_fix = ['name_score', 'location_distance', 'address_score']
for col in cols_to_fix:
    gplc_sf[col] = pd.to_numeric(gplc_sf[col], errors='coerce')
gplc_sf['address_score'] = gplc_sf['address_score'].fillna(0)
# gplc_sf['category_sim'] = gplc_sf['category_sim'].fillna(0)

#### Output the matched samples

In [198]:
df = gplc_sf[gplc_sf["matched_id"].notna()].copy()

N = 1000

weights = df["primary_cat"].map(
    df["primary_cat"].value_counts(normalize=True)
)

df_sample = df.sample(
    n=N,
    weights=weights,
    random_state=42
)
df_sample[['id','name','addr_simple','primary_type', 'matched_id','matched_name','matched_address','matched_cat_main','location_distance']].to_csv('/Users/houpuli/Redlining Lab Dropbox/HOUPU LI/POI research/Sioux_Falls_MSA/Sious_Falls_google_comparison/siousfalls_gplc_sf_1000_sample.csv', index=False)

#### Input the matched samples

In [ ]:
df_sample_check = pd.read_csv('/Users/houpuli/Redlining Lab Dropbox/HOUPU LI/POI research/ny_gplc_sf_2000_sample.csv')
df_sample_check = df_sample_check.drop(columns='location_distance')
df_sample_check = df_sample_check.merge(gplc_sf[['id','name_score','location_distance','address_score','category_sim']], left_on="id", right_on="id", how="left")

In [78]:
df_sample_check['is_true'].value_counts()

is_true
1    1840
0     160
Name: count, dtype: int64

In [80]:
df = df_sample_check.copy()
from sklearn.preprocessing import StandardScaler

X = df[['name_score', 'location_distance', 'address_score', 'category_sim']]
y = df['is_true']

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

In [81]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y,
    test_size=0.25,
    random_state=42,
    stratify=y  # keep the same proportion of True vs False in training set and test set
)

In [82]:
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import classification_report, roc_auc_score

gb_clf = GradientBoostingClassifier()
gb_clf.fit(X_train, y_train)

gb_y_pred = gb_clf.predict(X_test)
gb_y_prob = gb_clf.predict_proba(X_test)[:, 1]

gb_cr = classification_report(y_test, gb_y_pred)
gb_auc = roc_auc_score(y_test, gb_y_prob)

print(gb_cr)
print("GradientBoosting of AUC:", gb_auc)

              precision    recall  f1-score   support

           0       0.74      0.72      0.73        40
           1       0.98      0.98      0.98       460

    accuracy                           0.96       500
   macro avg       0.86      0.85      0.86       500
weighted avg       0.96      0.96      0.96       500

GradientBoosting of AUC: 0.9745652173913044


In [83]:
mask = ny_gplc_sf['matched_id'].notnull()
df_pred = ny_gplc_sf.loc[mask].copy()

feature_cols = ['name_score', 'location_distance', 'address_score', 'category_sim']

X_new = scaler.transform(df_pred[feature_cols])
df_pred['true_match_prob'] = gb_clf.predict_proba(X_new)[:, 1]
df_pred['is_true_match'] = df_pred['true_match_prob'] >= 0.5

ny_gplc_sf.loc[mask, 'is_true_match'] = df_pred['is_true_match']
ny_gplc_sf.loc[mask, 'true_match_prob'] = df_pred['true_match_prob']

In [84]:
ny_gplc_sf['is_true_match'].value_counts()

is_true_match
True     46717
False     4989
Name: count, dtype: int64

In [91]:
ny_gplc_sf.drop(columns=['cand_ids','cand_dist_m']).to_file('/Users/houpuli/Redlining Lab Dropbox/HOUPU LI/POI research/ny_gplc_sf.geojson')

# Google VS Foursquare

In [199]:
fsq = gpd.read_file('/Users/houpuli/Redlining Lab Dropbox/HOUPU LI/POI research/Sioux_Falls_MSA/Sious_Falls_msa_poi/siousfalls_fsq.geojson')
fsq

,fsq_place_id,name,latitude,longitude,address,locality,region,postcode,admin_region,post_town,...,fsq_category_ids,fsq_category_labels,placemaker_url,unresolved_flags,geom,bbox,cat_str,cat_main,cat_alt,geometry
0,62bde45e25ecb17924df322d,Exit 353,43.665276,-97.588120,NaN,Spencer,SD,57374,NaN,NaN,...,52f2ab2ebcbc57f1066b8b4c,Travel and Transportation > Road > Intersection,https://foursquare.com/placemakers/review-plac...,NaN,AAAAAAHAWGWjwLgwy0BF1SfDo1+P,"{'xmin': '-97.5881196783768', 'ymin': '43.6652...",Travel and Transportation > Road > Intersection,Travel and Transportation,Road,POINT (-97.58812 43.66528)
1,4e022400aeb76b61098a364b,Subway,43.663025,-97.587297,25678 431st Ave,Spencer,SD,57374,NaN,NaN,...,"4bf58dd8d48988d1c5941735, 4bf58dd8d48988d16e94...",Dining and Drinking > Restaurant > Sandwich Sp...,https://foursquare.com/placemakers/review-plac...,NaN,AAAAAAHAWGWWSDnUAEBF1N3+xrmK,"{'xmin': '-97.58729749343183', 'ymin': '43.663...",Dining and Drinking > Restaurant > Sandwich Spot,Dining and Drinking,Restaurant,POINT (-97.58730 43.66302)
2,4bd4d22a4e32d13a20febf80,Fuel Mart,43.663128,-97.587268,NaN,Emery,SD,57332,NaN,NaN,...,4bf58dd8d48988d113951735,Travel and Transportation > Fuel Station,https://foursquare.com/placemakers/review-plac...,NaN,AAAAAAHAWGWVywoM5EBF1OFfq9iO,"{'xmin': '-97.58726764661293', 'ymin': '43.663...",Travel and Transportation > Fuel Station,Travel and Transportation,Fuel Station,POINT (-97.58727 43.66313)
3,4ff12392e4b0279cf31e3676,Spencer,43.664962,-97.578009,NaN,Spencer,SD,57374,NaN,NaN,...,4f2a25ac4b909258e854f55f,Landmarks and Outdoors > States and Municipali...,https://foursquare.com/placemakers/review-plac...,NaN,AAAAAAHAWGT+GOxWaUBF1R15h8DJ,"{'xmin': '-97.57800887183988', 'ymin': '43.664...",Landmarks and Outdoors > States and Municipali...,Landmarks and Outdoors,States and Municipalities,POINT (-97.57801 43.66496)
4,4d4af7fd11a36ea814dd361c,Spencer AT&T Tower Site,43.664986,-97.537782,NaN,Spencer,SD,NaN,NaN,NaN,...,NaN,NaN,https://foursquare.com/placemakers/review-plac...,NaN,AAAAAAHAWGJrBunJkkBF1R5FxEzD,"{'xmin': '-97.5377824099617', 'ymin': '43.6649...",NaN,NaN,NaN,POINT (-97.53778 43.66499)
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
22281,f1950d6f567645f9310c45fa,Darwin Kirschenman Farm,43.138999,-97.472663,43617 293rd St,Menno,SD,57045,NaN,NaN,...,4bf58dd8d48988d15b941735,Landmarks and Outdoors > Farm,https://foursquare.com/placemakers/review-plac...,NaN,AAAAAAHAWF5AGmkOj0BFkcq7j0J1,"{'xmin': '-97.47266254672489', 'ymin': '43.138...",Landmarks and Outdoors > Farm,Landmarks and Outdoors,Farm,POINT (-97.47266 43.13900)
22282,1451b0852a254e0de2b03b16,Munkvold Land & Cattle Co.,43.124079,-97.454544,43723 294th St,Menno,SD,57045,NaN,NaN,...,NaN,NaN,https://foursquare.com/placemakers/review-plac...,NaN,AAAAAAHAWF0XPmGCVUBFj+HQ3kIK,"{'xmin': '-97.45454368135809', 'ymin': '43.124...",NaN,NaN,NaN,POINT (-97.45454 43.12408)
22283,536d58ad498ee2f0d55b2102,Jamesville,43.126017,-97.438841,NaN,Jamesville,SD,NaN,NaN,NaN,...,50aa9e094b90af0d42d5de0d,Landmarks and Outdoors > States and Municipali...,https://foursquare.com/placemakers/review-plac...,NaN,AAAAAAHAWFwV9sxqUkBFkCFTRhId,"{'xmin': '-97.4388405796283', 'ymin': '43.1260...",Landmarks and Outdoors > States and Municipali...,Landmarks and Outdoors,States and Municipalities,POINT (-97.43884 43.12602)
22284,8212d644987e483ff1e331a2,Jamesville Hutterian Brethren,43.101461,-97.480511,29568 436th Ave,Utica,SD,57067,NaN,NaN,...,63be6904847c3692a84b9b9a,Community and Government,https://foursquare.com/placemakers/review-plac...,NaN,AAAAAAHAWF7AsmUHdkBFjPysl0qm,"{'xmin': '-97.4805112825978', 'ymin': '43.1014...",Community and Government,Community and Government,NaN,POINT (-97.48051 43.10146)


In [200]:
gplc_fsq = search_spatial_candidates(reference_gdf=gplc, compared_gdf=fsq, id_col = "fsq_place_id", k=100)

In [201]:
gplc_fsq = match_by_name(reference_gdf=gplc_fsq, compared_gdf=fsq, re_name_col = "name", comp_name_col = "name", comp_id = "fsq_place_id", comp_id_col ="cat_alt",  threshold=80)

In [202]:
gplc_fsq = address_score_check(reference_gdf=gplc_fsq, compared_gdf=fsq, addr_col_ref = "addr_simple", addr_col_cmp = "address", id_col = "fsq_place_id")

In [ ]:
# dallas_gplc_fsq = calculate_similarity_check(dallas_gplc_fsq, cat_col_ref = "naics_definition", cat_col_cmp = "matched_cat_main", id_col = "matched_id", result_col =  "category_sim")

Encoding 39285 rows...


Batches:   0%|          | 0/154 [00:00<?, ?it/s]

Batches:   0%|          | 0/154 [00:00<?, ?it/s]

In [203]:
# transfer the X and Y into float type and deal with the address score
cols_to_fix = ['name_score', 'location_distance', 'address_score']
for col in cols_to_fix:
    gplc_fsq[col] = pd.to_numeric(gplc_fsq[col], errors='coerce')
gplc_fsq['address_score'] = gplc_fsq['address_score'].fillna(0)
# gplc_fsq['category_sim'] = gplc_fsq['category_sim'].fillna(0)

In [ ]:
df = gplc_fsq[gplc_fsq["matched_id"].notna()].copy()

N = 1000

weights = df["primary_cat"].map(
    df["primary_cat"].value_counts(normalize=True)
)

df_sample = df.sample(
    n=N,
    weights=weights,
    random_state=42
)
df_sample[['id','name','addr_simple','primary_type','matched_id','matched_name','matched_address','matched_cat_main','location_distance']].to_csv('/Users/houpuli/Redlining Lab Dropbox/HOUPU LI/POI research/Sioux_Falls_MSA/Sious_Falls_google_comparison/siousfalls_gplc_fsq_1000_sample.csv', index=False)

In [162]:
df_sample_check = pd.read_csv('/Users/houpuli/Redlining Lab Dropbox/HOUPU LI/POI research/ny_gplc_fsq_2000_sample.csv')
df_sample_check = df_sample_check.drop(columns='location_distance')
df_sample_check = df_sample_check.merge(ny_gplc_fsq[['id','name_score','location_distance','address_score','category_sim']], left_on="id", right_on="id", how="left")

In [163]:
df_sample_check['is_true'].value_counts()

is_true
1    1643
0     357
Name: count, dtype: int64

In [164]:
df = df_sample_check.copy()
from sklearn.preprocessing import StandardScaler

X = df[['name_score', 'location_distance', 'address_score', 'category_sim']]
y = df['is_true']

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

In [165]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y,
    test_size=0.25,
    random_state=42,
    stratify=y  # keep the same proportion of True vs False in training set and test set
)

In [166]:
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import classification_report, roc_auc_score

gb_clf = GradientBoostingClassifier()
gb_clf.fit(X_train, y_train)

gb_y_pred = gb_clf.predict(X_test)
gb_y_prob = gb_clf.predict_proba(X_test)[:, 1]

gb_cr = classification_report(y_test, gb_y_pred)
gb_auc = roc_auc_score(y_test, gb_y_prob)

print(gb_cr)
print("GradientBoosting of AUC:", gb_auc)

              precision    recall  f1-score   support

           0       0.84      0.87      0.85        89
           1       0.97      0.96      0.97       411

    accuracy                           0.95       500
   macro avg       0.90      0.91      0.91       500
weighted avg       0.95      0.95      0.95       500

GradientBoosting of AUC: 0.9787719729899671


In [167]:
mask = ny_gplc_fsq['matched_id'].notnull()
df_pred = ny_gplc_fsq.loc[mask].copy()

feature_cols = ['name_score', 'location_distance', 'address_score', 'category_sim']

X_new = scaler.transform(df_pred[feature_cols])
df_pred['true_match_prob'] = gb_clf.predict_proba(X_new)[:, 1]
df_pred['is_true_match'] = df_pred['true_match_prob'] >= 0.5

ny_gplc_fsq.loc[mask, 'is_true_match'] = df_pred['is_true_match']
ny_gplc_fsq.loc[mask, 'true_match_prob'] = df_pred['true_match_prob']

In [168]:
ny_gplc_fsq['is_true_match'].value_counts()

is_true_match
True     43745
False    10275
Name: count, dtype: int64

In [169]:
ny_gplc_fsq.drop(columns=['cand_ids','cand_dist_m']).to_file('/Users/houpuli/Redlining Lab Dropbox/HOUPU LI/POI research/ny_gplc_fsq.geojson')

# Google VS OSM

In [206]:
osm = gpd.read_file('/Users/houpuli/Redlining Lab Dropbox/HOUPU LI/POI research/Sioux_Falls_MSA/Sious_Falls_msa_poi/siousfalls_osm.geojson', driver="GeoJSON")

In [207]:
gplc_osm = search_spatial_candidates(reference_gdf=gplc, compared_gdf=osm, k=100)

In [208]:
gplc_osm = match_by_name(reference_gdf=gplc_osm, compared_gdf=osm, re_name_col = "name", comp_name_col = "name", comp_id_col ="cat", threshold=80)

In [209]:
gplc_osm = address_score_check(reference_gdf=gplc_osm, compared_gdf=osm, addr_col_ref = "addr_simple", addr_col_cmp = "address", id_col = "id")

In [ ]:
# gplc_osm = calculate_similarity_check(gplc_osm, cat_col_ref = "naics_definition", cat_col_cmp = "matched_cat_main", id_col = "matched_id", result_col =  "category_sim")

Encoding 22490 rows...


Batches:   0%|          | 0/88 [00:00<?, ?it/s]

Batches:   0%|          | 0/88 [00:00<?, ?it/s]

In [210]:
# transfer the X and Y into float type and deal with the address score
cols_to_fix = ['name_score', 'location_distance', 'address_score']
for col in cols_to_fix:
    gplc_osm[col] = pd.to_numeric(gplc_osm[col], errors='coerce')
gplc_osm['address_score'] = gplc_osm['address_score'].fillna(0)
# gplc_osm['category_sim'] = gplc_osm['category_sim'].fillna(0)

In [180]:
gplc_osm['matched_id'].notna().sum() / len(gplc_osm)

0.3269432202217435

#### Output the matched samples

In [211]:
df = gplc_osm[gplc_osm["matched_id"].notna()].copy()

N = 1000

weights = df["primary_cat"].map(
    df["primary_cat"].value_counts(normalize=True)
)

df_sample = df.sample(
    n=N,
    weights=weights,
    random_state=42
)
df_sample[['id','name','addr_simple','primary_type','matched_id','matched_name','matched_address','matched_cat_main','location_distance']].to_csv('/Users/houpuli/Redlining Lab Dropbox/HOUPU LI/POI research/Sioux_Falls_MSA/Sious_Falls_google_comparison/siousfalls_gplc_osm_1000_sample.csv', index=False)

#### Input the matched samples

In [223]:
df_sample_check = pd.read_csv('/Users/houpuli/Redlining Lab Dropbox/HOUPU LI/POI research/ny_gplc_osm_2000_sample.csv')
df_sample_check = df_sample_check.drop(columns='location_distance')
df_sample_check = df_sample_check.merge(ny_gplc_osm[['id','name_score','location_distance','address_score','category_sim']], left_on="id", right_on="id", how="left")

In [224]:
df_sample_check['is_true'].value_counts()

is_true
1    1415
0     585
Name: count, dtype: int64

In [225]:
df = df_sample_check.copy()
from sklearn.preprocessing import StandardScaler

X = df[['name_score', 'location_distance']]
y = df['is_true']

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

In [226]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y,
    test_size=0.25,
    random_state=42,
    stratify=y  # keep the same proportion of True vs False in training set and test set
)

In [227]:
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import classification_report, roc_auc_score

gb_clf = GradientBoostingClassifier()
gb_clf.fit(X_train, y_train)

gb_y_pred = gb_clf.predict(X_test)
gb_y_prob = gb_clf.predict_proba(X_test)[:, 1]

gb_cr = classification_report(y_test, gb_y_pred)
gb_auc = roc_auc_score(y_test, gb_y_prob)

print(gb_cr)
print("GradientBoosting of AUC:", gb_auc)

              precision    recall  f1-score   support

           0       0.84      0.95      0.89       146
           1       0.98      0.92      0.95       354

    accuracy                           0.93       500
   macro avg       0.91      0.93      0.92       500
weighted avg       0.94      0.93      0.93       500

GradientBoosting of AUC: 0.984356860924077


In [238]:
mask = ny_gplc_osm['matched_id'].notnull()
df_pred = ny_gplc_osm.loc[mask].copy()

feature_cols = ['name_score', 'location_distance']

X_new = scaler.transform(df_pred[feature_cols])
df_pred['true_match_prob'] = gb_clf.predict_proba(X_new)[:, 1]
df_pred['is_true_match'] = df_pred['true_match_prob'] >= 0.5

ny_gplc_osm.loc[mask, 'is_true_match'] = df_pred['is_true_match']
ny_gplc_osm.loc[mask, 'true_match_prob'] = df_pred['true_match_prob']

In [239]:
ny_gplc_osm['is_true_match'].value_counts()

is_true_match
True     19482
False     9574
Name: count, dtype: int64

In [242]:
ny_gplc_osm.drop(columns=['cand_ids','cand_dist_m']).to_file('/Users/houpuli/Redlining Lab Dropbox/HOUPU LI/POI research/ny_gplc_osm.geojson')